# FloodShift — Preprocessing

**Canonical notebook:** `03_FloodShift_Preprocessing`

This cleaned notebook contains one execution path only. Superseded DP-SSD normalization/DataLoader cells and emergency recovery cells have been removed.

**Execution rule:** run from top to bottom. The canonical FloodShift validation and test splits are not used for DP-SSD training or hyperparameter selection.


## 1. Canonical data connection and raw SAR audit


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C01
# SECTION: P3-S0 — SETUP
#
# PURPOSE:
#   Locate the frozen canonical raw FloodShift dataset.
#   Load the master manifest.
#
# NO preprocessing is performed in this cell.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json

# ------------------------------------------------------------
# Locate canonical manifest automatically
# ------------------------------------------------------------

INPUT_ROOT = Path("/kaggle/input")

manifest_candidates = list(
    INPUT_ROOT.rglob("master_manifest.csv")
)

if not manifest_candidates:
    raise FileNotFoundError(
        "master_manifest.csv not found.\n"
        "Attach floodshift-raw-canonical-v1 through Add Data."
    )

print("Found manifest candidates:")

for p in manifest_candidates:
    print(" ", p)

MANIFEST_PATH = manifest_candidates[0]

master = pd.read_csv(MANIFEST_PATH)

print("\n" + "=" * 72)
print("P3-C01 — CANONICAL DATASET CONNECTION")
print("=" * 72)

print("Manifest :", MANIFEST_PATH)
print("Scenes   :", len(master))

print("\nSplit counts:")
print(master["split"].value_counts())

print("\nRegions:")
print(master["region"].value_counts().sort_index())

assert len(master) == 440
assert master["scene_id"].is_unique

print("\nCanonical raw dataset connected ✅")


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C02
# SECTION: P3-S1 — RAW SAR UNDERSTANDING
#
# PURPOSE:
#   Inspect one Sentinel-1 scene BEFORE touching the data.
#
# Questions answered:
#   - How many bands?
#   - VV/VH value ranges?
#   - dtype?
#   - non-finite pixels?
#   - spatial dimensions?
# ============================================================

import rasterio
import numpy as np

# Use the first TRAIN scene only
sample = master[
    master["split"] == "train"
].iloc[0]

scene_id = sample["scene_id"]

# Important:
# Paths saved inside the old manifest may point to the
# previous /kaggle/working location.
#
# Therefore locate the file by filename inside /kaggle/input.

sar_filename = Path(
    sample["sar_raw_path"]
).name

sar_candidates = list(
    INPUT_ROOT.rglob(sar_filename)
)

if not sar_candidates:
    raise FileNotFoundError(
        f"Could not locate {sar_filename}"
    )

sar_path = sar_candidates[0]

with rasterio.open(sar_path) as src:

    sar = src.read()

    print("=" * 72)
    print("P3-C02 — RAW SENTINEL-1 INSPECTION")
    print("=" * 72)

    print("Scene      :", scene_id)
    print("Path       :", sar_path)
    print("Shape      :", sar.shape)
    print("Dtype      :", sar.dtype)
    print("CRS        :", src.crs)
    print("Resolution :", src.res)

print("\nOverall finite:")
print(np.isfinite(sar).all())

for band_idx, band_name in enumerate(["VV", "VH"]):

    x = sar[band_idx]

    finite = np.isfinite(x)

    print("\n" + "-" * 40)
    print(band_name)
    print("-" * 40)

    print("Finite pixels    :", finite.sum())
    print("Non-finite pixels:", (~finite).sum())

    if finite.any():

        values = x[finite]

        print("Minimum :", values.min())
        print("Maximum :", values.max())
        print("Mean    :", values.mean())
        print("Std     :", values.std())

        print(
            "Percentiles:",
            np.percentile(
                values,
                [0.5, 1, 5, 50, 95, 99, 99.5]
            )
        )


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C03
# SECTION: P3-S2 — SAR VALIDITY / NO-DATA AUDIT
#
# PURPOSE:
#   1. Check whether VV and VH share the same valid footprint.
#   2. Inspect GeoTIFF no-data metadata.
#   3. Compare SAR validity against the hand-label validity.
#   4. Create the canonical SAR validity mask.
#
# NO filtering is performed here.
# ============================================================

from pathlib import Path
import numpy as np
import rasterio

# ------------------------------------------------------------
# Resolve matching label file from canonical input
# ------------------------------------------------------------

label_filename = Path(
    sample["flood_label_path"]
).name

label_candidates = list(
    INPUT_ROOT.rglob(label_filename)
)

if not label_candidates:
    raise FileNotFoundError(
        f"Could not locate {label_filename}"
    )

label_path = label_candidates[0]


# ------------------------------------------------------------
# Read SAR + metadata
# ------------------------------------------------------------

with rasterio.open(sar_path) as src:

    sar = src.read()

    print("=" * 72)
    print("P3-C03 — SAR VALIDITY / NO-DATA AUDIT")
    print("=" * 72)

    print("Scene          :", scene_id)
    print("Raster nodata  :", src.nodata)
    print("Nodata values  :", src.nodatavals)
    print("Dataset mask   :", np.unique(src.dataset_mask(), return_counts=True))


# ------------------------------------------------------------
# Read hand label
# ------------------------------------------------------------

with rasterio.open(label_path) as src:
    label = src.read(1)

# ------------------------------------------------------------
# Construct masks
# ------------------------------------------------------------

vv_valid = np.isfinite(sar[0])
vh_valid = np.isfinite(sar[1])

# Canonical SAR pixel is valid only if BOTH polarizations exist
sar_valid = vv_valid & vh_valid

# Supervised-label validity
label_valid = label != -1


# ------------------------------------------------------------
# Compare VV and VH footprints
# ------------------------------------------------------------

same_footprint = np.array_equal(
    vv_valid,
    vh_valid
)

vv_only = vv_valid & ~vh_valid
vh_only = vh_valid & ~vv_valid


# ------------------------------------------------------------
# Compare SAR validity with supervision
# ------------------------------------------------------------

both_valid = sar_valid & label_valid

label_valid_sar_invalid = label_valid & ~sar_valid
sar_valid_label_invalid = sar_valid & ~label_valid


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

N = sar_valid.size

def pct(n):
    return 100.0 * n / N


print("\n" + "-" * 72)
print("POLARIZATION VALIDITY")
print("-" * 72)

print(
    f"VV valid              : {vv_valid.sum():,} "
    f"({pct(vv_valid.sum()):.2f}%)"
)

print(
    f"VH valid              : {vh_valid.sum():,} "
    f"({pct(vh_valid.sum()):.2f}%)"
)

print(
    f"Both VV + VH valid    : {sar_valid.sum():,} "
    f"({pct(sar_valid.sum()):.2f}%)"
)

print("Same VV/VH footprint  :", same_footprint)
print("VV-only valid pixels  :", int(vv_only.sum()))
print("VH-only valid pixels  :", int(vh_only.sum()))


print("\n" + "-" * 72)
print("SAR vs LABEL VALIDITY")
print("-" * 72)

print(
    f"Label-valid pixels          : {label_valid.sum():,} "
    f"({pct(label_valid.sum()):.2f}%)"
)

print(
    f"SAR + label both valid      : {both_valid.sum():,} "
    f"({pct(both_valid.sum()):.2f}%)"
)

print(
    f"Label valid, SAR invalid     : "
    f"{label_valid_sar_invalid.sum():,} "
    f"({pct(label_valid_sar_invalid.sum()):.2f}%)"
)

print(
    f"SAR valid, label invalid     : "
    f"{sar_valid_label_invalid.sum():,} "
    f"({pct(sar_valid_label_invalid.sum()):.2f}%)"
)


# ------------------------------------------------------------
# Safety assertions
# ------------------------------------------------------------

assert sar.shape == (2, 512, 512)

print("\nCanonical SAR validity mask created ✅")
print("sar_valid shape:", sar_valid.shape)
print("sar_valid dtype:", sar_valid.dtype)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C04
# SECTION: P3-S2 — SAFE dB <-> LINEAR CONVERSION
#
# PURPOSE:
#   Verify mathematically correct conversion between
#   Sentinel-1 dB values and linear intensity.
#
# IMPORTANT:
#   - Original NaN/no-data pixels remain NaN.
#   - No filtering is performed.
# ============================================================

import numpy as np

# ------------------------------------------------------------
# Conversion functions
# ------------------------------------------------------------

def db_to_linear(x_db, valid_mask=None):
    """
    Convert SAR dB intensity to linear intensity.

    Invalid positions remain NaN.
    """
    x_db = np.asarray(x_db, dtype=np.float32)

    if valid_mask is None:
        valid_mask = np.isfinite(x_db)

    out = np.full_like(
        x_db,
        np.nan,
        dtype=np.float32
    )

    out[valid_mask] = np.power(
        10.0,
        x_db[valid_mask] / 10.0
    )

    return out


def linear_to_db(x_linear, valid_mask=None, eps=1e-12):
    """
    Convert linear SAR intensity back to dB.

    Invalid positions remain NaN.
    """
    x_linear = np.asarray(
        x_linear,
        dtype=np.float32
    )

    if valid_mask is None:
        valid_mask = np.isfinite(x_linear)

    out = np.full_like(
        x_linear,
        np.nan,
        dtype=np.float32
    )

    out[valid_mask] = (
        10.0
        * np.log10(
            np.maximum(
                x_linear[valid_mask],
                eps
            )
        )
    )

    return out


# ------------------------------------------------------------
# Convert both polarizations
# ------------------------------------------------------------

sar_linear = np.empty_like(
    sar,
    dtype=np.float32
)

sar_roundtrip = np.empty_like(
    sar,
    dtype=np.float32
)

for band_idx in range(2):

    valid = sar_valid

    sar_linear[band_idx] = db_to_linear(
        sar[band_idx],
        valid
    )

    sar_roundtrip[band_idx] = linear_to_db(
        sar_linear[band_idx],
        valid
    )


# ------------------------------------------------------------
# Verify reconstruction accuracy
# ------------------------------------------------------------

print("=" * 72)
print("P3-C04 — dB ↔ LINEAR ROUND-TRIP TEST")
print("=" * 72)

for idx, name in enumerate(["VV", "VH"]):

    valid = sar_valid

    original = sar[idx][valid]
    restored = sar_roundtrip[idx][valid]

    abs_error = np.abs(
        original - restored
    )

    print(f"\n{name}")

    print(
        "Linear range:",
        float(np.nanmin(sar_linear[idx])),
        "to",
        float(np.nanmax(sar_linear[idx]))
    )

    print(
        "Max round-trip error :",
        float(abs_error.max())
    )

    print(
        "Mean round-trip error:",
        float(abs_error.mean())
    )

    print(
        "Invalid pixels preserved:",
        np.isnan(
            sar_roundtrip[idx][~valid]
        ).all()
    )


# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

assert np.isnan(
    sar_roundtrip[:, ~sar_valid]
).all()

assert np.allclose(
    sar[:, sar_valid],
    sar_roundtrip[:, sar_valid],
    atol=1e-5
)

print("\nSafe dB ↔ linear conversion verified ✅")


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C05
# SECTION: P3-S3 — TRAIN SAR DISTRIBUTION AUDIT
#
# PURPOSE:
#   Characterize raw Sentinel-1 VV/VH across TRAIN scenes only.
#
#   We measure:
#   - valid pixel coverage
#   - non-finite coverage
#   - global train distribution
#   - robust percentiles
#
# IMPORTANT:
#   No validation/test statistics are used.
#   No filtering is performed.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import rasterio

train_df = master[master["split"] == "train"].copy()

print("Training scenes:", len(train_df))

# ------------------------------------------------------------
# Helper: locate file inside attached Kaggle input
# ------------------------------------------------------------

def locate_input_file(saved_path):
    filename = Path(saved_path).name
    matches = list(INPUT_ROOT.rglob(filename))

    if not matches:
        raise FileNotFoundError(filename)

    return matches[0]


# ------------------------------------------------------------
# Collect statistics
# ------------------------------------------------------------

scene_rows = []

vv_chunks = []
vh_chunks = []

# To avoid storing every pixel from all 314 scenes,
# sample a maximum number of valid pixels per scene.
MAX_PIXELS_PER_SCENE = 50_000

rng = np.random.default_rng(42)

for idx, row in train_df.iterrows():

    sar_path_i = locate_input_file(
        row["sar_raw_path"]
    )

    with rasterio.open(sar_path_i) as src:
        x = src.read().astype(np.float32)

    vv = x[0]
    vh = x[1]

    valid = (
        np.isfinite(vv)
        &
        np.isfinite(vh)
    )

    valid_count = int(valid.sum())
    total_count = int(valid.size)

    if valid_count > 0:

        vv_valid = vv[valid]
        vh_valid = vh[valid]

        # --------------------------------------------
        # Random subsample for global distribution
        # --------------------------------------------

        if valid_count > MAX_PIXELS_PER_SCENE:

            sample_idx = rng.choice(
                valid_count,
                size=MAX_PIXELS_PER_SCENE,
                replace=False
            )

            vv_sample = vv_valid[sample_idx]
            vh_sample = vh_valid[sample_idx]

        else:

            vv_sample = vv_valid
            vh_sample = vh_valid

        vv_chunks.append(vv_sample)
        vh_chunks.append(vh_sample)

        vv_min = float(vv_valid.min())
        vv_max = float(vv_valid.max())
        vv_mean = float(vv_valid.mean())
        vv_std = float(vv_valid.std())

        vh_min = float(vh_valid.min())
        vh_max = float(vh_valid.max())
        vh_mean = float(vh_valid.mean())
        vh_std = float(vh_valid.std())

    else:

        vv_min = vv_max = vv_mean = vv_std = np.nan
        vh_min = vh_max = vh_mean = vh_std = np.nan

    scene_rows.append({
        "scene_id": row["scene_id"],
        "region": row["region"],

        "valid_pixels": valid_count,
        "invalid_pixels": total_count - valid_count,
        "valid_pct": 100.0 * valid_count / total_count,

        "vv_min_db": vv_min,
        "vv_max_db": vv_max,
        "vv_mean_db": vv_mean,
        "vv_std_db": vv_std,

        "vh_min_db": vh_min,
        "vh_max_db": vh_max,
        "vh_mean_db": vh_mean,
        "vh_std_db": vh_std,
    })


scene_stats = pd.DataFrame(scene_rows)

vv_global = np.concatenate(vv_chunks)
vh_global = np.concatenate(vh_chunks)


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("P3-C05 — TRAIN SAR DISTRIBUTION AUDIT")
print("=" * 72)

print("\nVALID COVERAGE")
print("-" * 72)

print(
    scene_stats["valid_pct"].describe(
        percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    )
)

print("\nScenes with < 25% SAR coverage:",
      int((scene_stats["valid_pct"] < 25).sum()))

print("Scenes with < 50% SAR coverage:",
      int((scene_stats["valid_pct"] < 50).sum()))

print("Scenes with 100% SAR coverage:",
      int((scene_stats["valid_pct"] == 100).sum()))


# ------------------------------------------------------------
# Robust global distribution
# ------------------------------------------------------------

percentiles = [
    0.1,
    0.5,
    1,
    2,
    5,
    25,
    50,
    75,
    95,
    98,
    99,
    99.5,
    99.9
]

print("\nVV TRAIN DISTRIBUTION [dB]")
print("-" * 72)

for p, v in zip(
    percentiles,
    np.percentile(vv_global, percentiles)
):
    print(f"{p:>5.1f}% : {v:>10.4f}")


print("\nVH TRAIN DISTRIBUTION [dB]")
print("-" * 72)

for p, v in zip(
    percentiles,
    np.percentile(vh_global, percentiles)
):
    print(f"{p:>5.1f}% : {v:>10.4f}")


print("\nGLOBAL SAMPLE SIZE")
print("-" * 72)

print("VV sampled pixels:", f"{len(vv_global):,}")
print("VH sampled pixels:", f"{len(vh_global):,}")


# ------------------------------------------------------------
# Save audit locally in this notebook
# ------------------------------------------------------------

OUTPUT_ROOT = Path(
    "/kaggle/working/FloodShift_Preprocessing"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

scene_stats.to_csv(
    OUTPUT_ROOT / "train_raw_sar_scene_statistics.csv",
    index=False
)

print(
    "\nSaved:",
    OUTPUT_ROOT / "train_raw_sar_scene_statistics.csv"
)

print("\nTRAIN-only SAR audit complete ✅")


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C06
# SECTION: P3-S3 — REPRESENTATIVE RAW SAR VISUALIZATION
#
# PURPOSE:
#   Inspect representative training scenes before despeckling.
#
#   Automatically selects:
#     1. low-texture scene
#     2. medium-texture scene
#     3. high-texture scene
#
#   Selection uses SAR statistics only — NOT labels.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import rasterio

# ------------------------------------------------------------
# Use only scenes with complete SAR coverage
# ------------------------------------------------------------

full_coverage = scene_stats[
    scene_stats["valid_pct"] == 100
].copy()

# VV standard deviation is used only as a simple
# scene-level texture proxy.
texture_values = full_coverage["vv_std_db"].values

targets = {
    "LOW texture": np.percentile(texture_values, 10),
    "MEDIUM texture": np.percentile(texture_values, 50),
    "HIGH texture": np.percentile(texture_values, 90),
}

selected_rows = []

for label_name, target_std in targets.items():

    temp = full_coverage.copy()

    temp["distance"] = np.abs(
        temp["vv_std_db"] - target_std
    )

    row = temp.sort_values(
        "distance"
    ).iloc[0]

    selected_rows.append(
        (label_name, row)
    )


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

fig, axes = plt.subplots(
    len(selected_rows),
    2,
    figsize=(12, 14)
)

for i, (category, row) in enumerate(selected_rows):

    original_row = train_df[
        train_df["scene_id"] == row["scene_id"]
    ].iloc[0]

    path = locate_input_file(
        original_row["sar_raw_path"]
    )

    with rasterio.open(path) as src:
        x = src.read().astype(np.float32)

    vv = x[0]
    vh = x[1]

    # --------------------------------------------------------
    # Display limits only
    #
    # These DO NOT alter the underlying SAR.
    # Robust percentiles simply improve visualization.
    # --------------------------------------------------------

    vv_valid = vv[np.isfinite(vv)]
    vh_valid = vh[np.isfinite(vh)]

    vv_lo, vv_hi = np.percentile(
        vv_valid,
        [2, 98]
    )

    vh_lo, vh_hi = np.percentile(
        vh_valid,
        [2, 98]
    )

    axes[i, 0].imshow(
        vv,
        cmap="gray",
        vmin=vv_lo,
        vmax=vv_hi
    )

    axes[i, 0].set_title(
        f"{category} — {row['scene_id']} — VV\n"
        f"scene std = {row['vv_std_db']:.2f} dB"
    )

    axes[i, 0].axis("off")

    axes[i, 1].imshow(
        vh,
        cmap="gray",
        vmin=vh_lo,
        vmax=vh_hi
    )

    axes[i, 1].set_title(
        f"{category} — {row['scene_id']} — VH\n"
        f"scene std = {row['vh_std_db']:.2f} dB"
    )

    axes[i, 1].axis("off")


plt.tight_layout()
plt.show()


print("=" * 72)
print("P3-C06 — REPRESENTATIVE SCENES")
print("=" * 72)

for category, row in selected_rows:

    print(
        f"{category:<16} "
        f"{row['scene_id']:<25} "
        f"VV std={row['vv_std_db']:.3f} dB   "
        f"VH std={row['vh_std_db']:.3f} dB"
    )

print("\nRaw SAR visualization complete ✅")


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C07
# SECTION: P3-S3 — AUTOMATIC HOMOGENEOUS PATCH SELECTION
#
# PURPOSE:
#   Automatically identify low-variation SAR regions that can
#   later be reused for objective speckle evaluation.
#
# IMPORTANT:
#   - Uses SAR only.
#   - Uses TRAIN scenes only.
#   - Does NOT use flood labels.
#   - Patch coordinates are frozen once selected.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

PATCH_SIZE = 32
STRIDE = 8

patch_records = []


def patch_statistics(x_linear):
    """
    Calculate mean, std, coefficient of variation and ENL
    for a finite linear-intensity patch.
    """

    values = x_linear[np.isfinite(x_linear)]

    if len(values) == 0:
        return None

    mean = float(values.mean())
    std = float(values.std())

    if mean <= 0:
        return None

    cv = std / mean

    enl = (
        (mean ** 2) / (std ** 2)
        if std > 0
        else np.inf
    )

    return mean, std, cv, enl


# ------------------------------------------------------------
# Search each representative scene
# ------------------------------------------------------------

for category, stats_row in selected_rows:

    scene_id_i = stats_row["scene_id"]

    manifest_row = train_df[
        train_df["scene_id"] == scene_id_i
    ].iloc[0]

    sar_path_i = locate_input_file(
        manifest_row["sar_raw_path"]
    )

    with rasterio.open(sar_path_i) as src:
        x_db = src.read().astype(np.float32)

    vv_db = x_db[0]
    vh_db = x_db[1]

    valid = (
        np.isfinite(vv_db)
        &
        np.isfinite(vh_db)
    )

    # --------------------------------------------------------
    # Convert to linear intensity
    # --------------------------------------------------------

    vv_linear = db_to_linear(
        vv_db,
        valid
    )

    vh_linear = db_to_linear(
        vh_db,
        valid
    )

    candidates = []

    H, W = valid.shape

    for y in range(
        0,
        H - PATCH_SIZE + 1,
        STRIDE
    ):

        for x in range(
            0,
            W - PATCH_SIZE + 1,
            STRIDE
        ):

            valid_patch = valid[
                y:y + PATCH_SIZE,
                x:x + PATCH_SIZE
            ]

            # Require complete valid SAR coverage
            if not valid_patch.all():
                continue

            vv_patch = vv_linear[
                y:y + PATCH_SIZE,
                x:x + PATCH_SIZE
            ]

            vh_patch = vh_linear[
                y:y + PATCH_SIZE,
                x:x + PATCH_SIZE
            ]

            vv_stats = patch_statistics(vv_patch)
            vh_stats = patch_statistics(vh_patch)

            if vv_stats is None or vh_stats is None:
                continue

            vv_mean, vv_std, vv_cv, vv_enl = vv_stats
            vh_mean, vh_std, vh_cv, vh_enl = vh_stats

            # ------------------------------------------------
            # Combined score:
            # low CV in BOTH VV and VH is preferred.
            # ------------------------------------------------

            score = (vv_cv + vh_cv) / 2.0

            candidates.append({
                "scene_id": scene_id_i,
                "category": category,

                "x": x,
                "y": y,
                "patch_size": PATCH_SIZE,

                "vv_cv": vv_cv,
                "vh_cv": vh_cv,

                "vv_enl_raw": vv_enl,
                "vh_enl_raw": vh_enl,

                "score": score,
            })

    if not candidates:
        raise RuntimeError(
            f"No valid homogeneous patch found for {scene_id_i}"
        )

    candidates_df = pd.DataFrame(candidates)

    # Lowest combined CV = most homogeneous candidate
    best = candidates_df.sort_values(
        "score"
    ).iloc[0]

    patch_records.append(
        best.to_dict()
    )


# ------------------------------------------------------------
# Final frozen patch table
# ------------------------------------------------------------

homogeneous_patches = pd.DataFrame(
    patch_records
)

print("=" * 72)
print("P3-C07 — AUTOMATIC HOMOGENEOUS PATCHES")
print("=" * 72)

print(
    homogeneous_patches[
        [
            "category",
            "scene_id",
            "x",
            "y",
            "patch_size",
            "vv_cv",
            "vh_cv",
            "vv_enl_raw",
            "vh_enl_raw",
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# Visual verification
# ------------------------------------------------------------

fig, axes = plt.subplots(
    len(homogeneous_patches),
    2,
    figsize=(11, 13)
)

for i, row in homogeneous_patches.iterrows():

    scene_id_i = row["scene_id"]

    manifest_row = train_df[
        train_df["scene_id"] == scene_id_i
    ].iloc[0]

    sar_path_i = locate_input_file(
        manifest_row["sar_raw_path"]
    )

    with rasterio.open(sar_path_i) as src:
        x_db = src.read().astype(np.float32)

    x0 = int(row["x"])
    y0 = int(row["y"])
    ps = int(row["patch_size"])

    for band_idx, band_name in enumerate(
        ["VV", "VH"]
    ):

        band = x_db[band_idx]

        finite = band[np.isfinite(band)]

        lo, hi = np.percentile(
            finite,
            [2, 98]
        )

        ax = axes[i, band_idx]

        ax.imshow(
            band,
            cmap="gray",
            vmin=lo,
            vmax=hi
        )

        rect = plt.Rectangle(
            (x0, y0),
            ps,
            ps,
            fill=False,
            linewidth=2
        )

        ax.add_patch(rect)

        ax.set_title(
            f"{row['category']} — {scene_id_i}\n"
            f"{band_name} homogeneous patch"
        )

        ax.axis("off")

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# Save frozen patch coordinates
# ------------------------------------------------------------

patch_file = (
    OUTPUT_ROOT /
    "sar_homogeneous_evaluation_patches.csv"
)

homogeneous_patches.to_csv(
    patch_file,
    index=False
)

print("\nSaved:", patch_file)

print(
    "\nThese coordinates are now reusable across "
    "all SAR preprocessing methods ✅"
)


## 2. Existing SAR despeckling baselines


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C08
# SECTION: P3-S4 — REFINED LEE DESPECKLING
#
# PURPOSE:
#   1. Implement a directional Refined Lee filter.
#   2. Apply it in LINEAR intensity space.
#   3. Convert the result back to dB.
#   4. Compare Raw vs Refined Lee on the SAME frozen
#      homogeneous evaluation patches.
#
# IMPORTANT:
#   - TRAIN scenes only.
#   - No labels used.
#   - No normalization yet.
#   - No files are permanently generated yet.
# ============================================================

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

from scipy.ndimage import (
    uniform_filter,
    convolve
)


# ------------------------------------------------------------
# Helper: reflect-safe spatial shift
# ------------------------------------------------------------

def shift_reflect(arr, dy, dx):
    """
    Shift array spatially without wrap-around.
    Reflective padding is used at boundaries.
    """

    H, W = arr.shape

    pad = max(
        abs(dy),
        abs(dx)
    )

    padded = np.pad(
        arr,
        pad_width=pad,
        mode="reflect"
    )

    y0 = pad + dy
    x0 = pad + dx

    return padded[
        y0:y0 + H,
        x0:x0 + W
    ]


# ------------------------------------------------------------
# Refined Lee
# ------------------------------------------------------------

def refined_lee_filter(
    image_linear,
    eps=1e-12
):
    """
    Directional Refined Lee filter for a positive
    SAR linear-intensity image.

    Input:
        image_linear : 2D linear-intensity SAR

    Returns:
        filtered linear-intensity SAR
    """

    img = np.asarray(
        image_linear,
        dtype=np.float64
    )

    if not np.isfinite(img).all():
        raise ValueError(
            "refined_lee_filter currently expects "
            "a fully valid test image."
        )

    # --------------------------------------------------------
    # STEP 1
    # 3x3 local mean and variance
    # --------------------------------------------------------

    mean3 = uniform_filter(
        img,
        size=3,
        mode="reflect"
    )

    mean_sq3 = uniform_filter(
        img ** 2,
        size=3,
        mode="reflect"
    )

    var3 = np.maximum(
        mean_sq3 - mean3 ** 2,
        0.0
    )


    # --------------------------------------------------------
    # STEP 2
    # Sample the 3x3 statistics at 9 locations distributed
    # across the surrounding 7x7 neighborhood.
    #
    # Layout:
    #
    #   0  1  2
    #   3  4  5
    #   6  7  8
    #
    # --------------------------------------------------------

    offsets = [
        (-2, -2), (-2,  0), (-2,  2),
        ( 0, -2), ( 0,  0), ( 0,  2),
        ( 2, -2), ( 2,  0), ( 2,  2),
    ]

    sample_mean = np.stack(
        [
            shift_reflect(
                mean3,
                dy,
                dx
            )
            for dy, dx in offsets
        ],
        axis=0
    )

    sample_var = np.stack(
        [
            shift_reflect(
                var3,
                dy,
                dx
            )
            for dy, dx in offsets
        ],
        axis=0
    )


    # --------------------------------------------------------
    # STEP 3
    # Four principal gradient magnitudes.
    # --------------------------------------------------------

    gradients = np.stack(
        [
            np.abs(
                sample_mean[1]
                -
                sample_mean[7]
            ),

            np.abs(
                sample_mean[6]
                -
                sample_mean[2]
            ),

            np.abs(
                sample_mean[3]
                -
                sample_mean[5]
            ),

            np.abs(
                sample_mean[0]
                -
                sample_mean[8]
            ),
        ],
        axis=0
    )

    max_gradient = np.argmax(
        gradients,
        axis=0
    )


    # --------------------------------------------------------
    # STEP 4
    # Determine one of 8 local directions.
    # --------------------------------------------------------

    d1 = (
        sample_mean[1] - sample_mean[4]
        >
        sample_mean[4] - sample_mean[7]
    )

    d2 = (
        sample_mean[6] - sample_mean[4]
        >
        sample_mean[4] - sample_mean[2]
    )

    d3 = (
        sample_mean[3] - sample_mean[4]
        >
        sample_mean[4] - sample_mean[5]
    )

    d4 = (
        sample_mean[0] - sample_mean[4]
        >
        sample_mean[4] - sample_mean[8]
    )

    first_directions = np.stack(
        [
            np.where(d1, 1, 5),
            np.where(d2, 2, 6),
            np.where(d3, 3, 7),
            np.where(d4, 4, 8),
        ],
        axis=0
    )

    yy, xx = np.indices(
        img.shape
    )

    direction = first_directions[
        max_gradient,
        yy,
        xx
    ]


    # --------------------------------------------------------
    # STEP 5
    # Estimate local speckle variance.
    #
    # sigmaV is estimated from the five lowest
    # normalized local variances among the 9 samples.
    # --------------------------------------------------------

    normalized_variance = (
        sample_var
        /
        (
            sample_mean ** 2
            +
            eps
        )
    )

    normalized_sorted = np.sort(
        normalized_variance,
        axis=0
    )

    sigmaV = np.mean(
        normalized_sorted[:5],
        axis=0
    )


    # --------------------------------------------------------
    # STEP 6
    # Directional kernels
    # --------------------------------------------------------

    rect_kernel = np.array(
        [
            [0,0,0,0,0,0,0],
            [0,0,0,0,0,0,0],
            [0,0,0,0,0,0,0],
            [1,1,1,1,1,1,1],
            [1,1,1,1,1,1,1],
            [1,1,1,1,1,1,1],
            [1,1,1,1,1,1,1],
        ],
        dtype=np.float64
    )

    diag_kernel = np.array(
        [
            [1,0,0,0,0,0,0],
            [1,1,0,0,0,0,0],
            [1,1,1,0,0,0,0],
            [1,1,1,1,0,0,0],
            [1,1,1,1,1,0,0],
            [1,1,1,1,1,1,0],
            [1,1,1,1,1,1,1],
        ],
        dtype=np.float64
    )

    kernels = []

    for rot in range(4):

        kernels.append(
            np.rot90(
                rect_kernel,
                rot
            )
        )

        kernels.append(
            np.rot90(
                diag_kernel,
                rot
            )
        )


    # --------------------------------------------------------
    # STEP 7
    # Calculate directional mean/variance.
    # --------------------------------------------------------

    dir_mean = np.zeros_like(
        img
    )

    dir_var = np.zeros_like(
        img
    )

    for direction_id, kernel in enumerate(
        kernels,
        start=1
    ):

        weight = kernel.sum()

        local_mean = convolve(
            img,
            kernel,
            mode="reflect"
        ) / weight

        local_mean_sq = convolve(
            img ** 2,
            kernel,
            mode="reflect"
        ) / weight

        local_var = np.maximum(
            local_mean_sq
            -
            local_mean ** 2,
            0.0
        )

        mask = (
            direction
            ==
            direction_id
        )

        dir_mean[mask] = local_mean[mask]
        dir_var[mask] = local_var[mask]


    # --------------------------------------------------------
    # STEP 8
    # Refined Lee adaptive weighting
    # --------------------------------------------------------

    var_x = (
        dir_var
        -
        (
            dir_mean ** 2
            *
            sigmaV
        )
    ) / (
        sigmaV
        +
        1.0
    )

    var_x = np.maximum(
        var_x,
        0.0
    )

    b = (
        var_x
        /
        (
            dir_var
            +
            eps
        )
    )

    b = np.clip(
        b,
        0.0,
        1.0
    )

    filtered = (
        dir_mean
        +
        b
        *
        (
            img
            -
            dir_mean
        )
    )

    filtered = np.maximum(
        filtered,
        eps
    )

    return filtered.astype(
        np.float32
    )


# ============================================================
# APPLY TO THE THREE FROZEN REFERENCE SCENES
# ============================================================

comparison_records = []

visual_records = []


for _, patch_row in homogeneous_patches.iterrows():

    scene_id_i = patch_row["scene_id"]

    manifest_row = train_df[
        train_df["scene_id"]
        ==
        scene_id_i
    ].iloc[0]

    path = locate_input_file(
        manifest_row["sar_raw_path"]
    )

    with rasterio.open(path) as src:

        raw_db = src.read().astype(
            np.float32
        )

    # These three reference scenes were selected
    # specifically from complete-coverage scenes.
    valid = (
        np.isfinite(raw_db[0])
        &
        np.isfinite(raw_db[1])
    )

    assert valid.all(), (
        f"{scene_id_i} unexpectedly contains invalid SAR."
    )

    filtered_db = np.empty_like(
        raw_db
    )


    for band_idx in range(2):

        # dB → linear
        linear = db_to_linear(
            raw_db[band_idx],
            valid
        )

        # Refined Lee in linear domain
        filtered_linear = refined_lee_filter(
            linear
        )

        # linear → dB
        filtered_db[band_idx] = linear_to_db(
            filtered_linear,
            valid
        )


    # --------------------------------------------------------
    # SAME frozen homogeneous patch
    # --------------------------------------------------------

    x0 = int(patch_row["x"])
    y0 = int(patch_row["y"])
    ps = int(patch_row["patch_size"])

    record = {
        "scene_id": scene_id_i,
        "category": patch_row["category"]
    }


    for band_idx, band_name in enumerate(
        ["VV", "VH"]
    ):

        raw_patch_db = raw_db[
            band_idx,
            y0:y0+ps,
            x0:x0+ps
        ]

        lee_patch_db = filtered_db[
            band_idx,
            y0:y0+ps,
            x0:x0+ps
        ]

        # ENL must be calculated in linear intensity.
        raw_patch_linear = np.power(
            10.0,
            raw_patch_db / 10.0
        )

        lee_patch_linear = np.power(
            10.0,
            lee_patch_db / 10.0
        )

        raw_stats = patch_statistics(
            raw_patch_linear
        )

        lee_stats = patch_statistics(
            lee_patch_linear
        )

        raw_enl = raw_stats[3]
        lee_enl = lee_stats[3]

        record[
            f"{band_name.lower()}_raw_enl"
        ] = raw_enl

        record[
            f"{band_name.lower()}_refined_lee_enl"
        ] = lee_enl

        record[
            f"{band_name.lower()}_enl_gain"
        ] = lee_enl / raw_enl


    comparison_records.append(
        record
    )

    visual_records.append(
        (
            scene_id_i,
            patch_row["category"],
            raw_db,
            filtered_db
        )
    )


refined_lee_results = pd.DataFrame(
    comparison_records
)


# ============================================================
# REPORT
# ============================================================

print("=" * 78)
print("P3-C08 — REFINED LEE FIRST EVALUATION")
print("=" * 78)

print(
    refined_lee_results.to_string(
        index=False
    )
)


# ============================================================
# VISUAL COMPARISON
# ============================================================

fig, axes = plt.subplots(
    len(visual_records),
    4,
    figsize=(16, 13)
)


for i, (
    scene_id_i,
    category,
    raw_db,
    filtered_db
) in enumerate(visual_records):

    for band_idx, band_name in enumerate(
        ["VV", "VH"]
    ):

        raw_band = raw_db[band_idx]

        finite = raw_band[
            np.isfinite(raw_band)
        ]

        lo, hi = np.percentile(
            finite,
            [2, 98]
        )

        # RAW
        ax_raw = axes[
            i,
            band_idx * 2
        ]

        ax_raw.imshow(
            raw_band,
            cmap="gray",
            vmin=lo,
            vmax=hi
        )

        ax_raw.set_title(
            f"{category}\n"
            f"{scene_id_i} {band_name} RAW"
        )

        ax_raw.axis("off")


        # REFINED LEE
        ax_lee = axes[
            i,
            band_idx * 2 + 1
        ]

        ax_lee.imshow(
            filtered_db[band_idx],
            cmap="gray",
            vmin=lo,
            vmax=hi
        )

        ax_lee.set_title(
            f"{category}\n"
            f"{scene_id_i} {band_name} Refined Lee"
        )

        ax_lee.axis("off")


plt.tight_layout()
plt.show()


print(
    "\nRefined Lee test completed ✅"
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C09
# SECTION: P3-S4 — REFINED LEE EDGE PRESERVATION
#
# PURPOSE:
#   Quantify whether Refined Lee preserves strong SAR structure.
#
# METRICS:
#   EGR = Edge Gradient Retention
#   EGC = Edge Gradient Correlation
#
# IMPORTANT:
#   - Same three frozen TRAIN scenes.
#   - No labels used.
#   - Gaussian smoothing is ONLY for locating structural edges.
# ============================================================

import numpy as np
import pandas as pd

from scipy.ndimage import (
    gaussian_filter,
    sobel
)


# ------------------------------------------------------------
# Gradient magnitude
# ------------------------------------------------------------

def gradient_magnitude(image):
    gx = sobel(
        image,
        axis=1,
        mode="reflect"
    )

    gy = sobel(
        image,
        axis=0,
        mode="reflect"
    )

    return np.sqrt(
        gx**2 + gy**2
    )


def edge_preservation_metrics(
    raw_db,
    filtered_db,
    edge_percentile=90
):
    """
    Evaluate preservation of strong structural edges.

    Returns:
        EGR  - gradient strength retention
        EGC  - gradient correlation
        n_edges
    """

    # --------------------------------------------------------
    # Light smoothing ONLY to identify reliable structures.
    # This suppresses tiny speckle-induced gradients.
    # --------------------------------------------------------

    raw_for_edges = gaussian_filter(
        raw_db,
        sigma=1.0,
        mode="reflect"
    )

    structural_gradient = gradient_magnitude(
        raw_for_edges
    )

    threshold = np.percentile(
        structural_gradient,
        edge_percentile
    )

    edge_mask = (
        structural_gradient >= threshold
    )


    # --------------------------------------------------------
    # Gradients of actual raw + filtered images
    # --------------------------------------------------------

    raw_gradient = gradient_magnitude(
        raw_db
    )

    filtered_gradient = gradient_magnitude(
        filtered_db
    )


    raw_edge_grad = raw_gradient[
        edge_mask
    ]

    filtered_edge_grad = filtered_gradient[
        edge_mask
    ]


    # --------------------------------------------------------
    # Edge Gradient Retention
    # --------------------------------------------------------

    egr = (
        filtered_edge_grad.sum()
        /
        (raw_edge_grad.sum() + 1e-12)
    )


    # --------------------------------------------------------
    # Edge Gradient Correlation
    # --------------------------------------------------------

    if (
        raw_edge_grad.std() > 0
        and
        filtered_edge_grad.std() > 0
    ):

        egc = np.corrcoef(
            raw_edge_grad,
            filtered_edge_grad
        )[0, 1]

    else:

        egc = np.nan


    return {
        "egr": float(egr),
        "egc": float(egc),
        "edge_pixels": int(edge_mask.sum())
    }


# ============================================================
# CALCULATE FOR THE SAME THREE REFERENCE SCENES
# ============================================================

edge_records = []

for (
    scene_id_i,
    category,
    raw_db,
    filtered_db
) in visual_records:

    record = {
        "scene_id": scene_id_i,
        "category": category
    }

    for band_idx, band_name in enumerate(
        ["VV", "VH"]
    ):

        metrics = edge_preservation_metrics(
            raw_db[band_idx],
            filtered_db[band_idx]
        )

        record[
            f"{band_name.lower()}_egr"
        ] = metrics["egr"]

        record[
            f"{band_name.lower()}_egc"
        ] = metrics["egc"]

        record[
            f"{band_name.lower()}_edge_pixels"
        ] = metrics["edge_pixels"]


    edge_records.append(
        record
    )


refined_lee_edge_results = pd.DataFrame(
    edge_records
)


# ============================================================
# REPORT
# ============================================================

print("=" * 78)
print("P3-C09 — REFINED LEE EDGE PRESERVATION")
print("=" * 78)

print(
    refined_lee_edge_results.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nSUMMARY")
print("-" * 78)

print(
    "Mean VV EGR:",
    refined_lee_edge_results[
        "vv_egr"
    ].mean()
)

print(
    "Mean VV EGC:",
    refined_lee_edge_results[
        "vv_egc"
    ].mean()
)

print(
    "Mean VH EGR:",
    refined_lee_edge_results[
        "vh_egr"
    ].mean()
)

print(
    "Mean VH EGC:",
    refined_lee_edge_results[
        "vh_egc"
    ].mean()
)

print(
    "\nRefined Lee edge-preservation audit complete ✅"
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C10
# SECTION: P3-S5 — LEE SIGMA DESPECKLING
#
# PURPOSE:
#   Compare Lee Sigma against Raw and Refined Lee using:
#       - same scenes
#       - same homogeneous patches
#       - same ENL metric
#       - same edge-preservation metric
#
# IMPORTANT:
#   - Processing is performed in LINEAR intensity.
#   - No labels are used.
#   - No normalization yet.
# ============================================================

import numpy as np
import pandas as pd
import rasterio

from scipy.ndimage import uniform_filter


# ------------------------------------------------------------
# Lee Sigma filter
# ------------------------------------------------------------

def lee_sigma_filter(
    image_linear,
    window_size=7,
    sigma_factor=0.9,
    eps=1e-12
):
    """
    Practical Lee-Sigma implementation.

    Steps:
      1. estimate local mean/std
      2. retain neighbors lying inside a sigma interval
      3. calculate statistics from selected pixels
      4. apply adaptive Lee/MMSE weighting

    Input must be positive linear SAR intensity.
    """

    img = np.asarray(
        image_linear,
        dtype=np.float64
    )

    if not np.isfinite(img).all():
        raise ValueError(
            "Lee Sigma test expects a fully valid image."
        )

    radius = window_size // 2

    H, W = img.shape

    padded = np.pad(
        img,
        radius,
        mode="reflect"
    )

    output = np.empty_like(img)

    # --------------------------------------------------------
    # Initial local statistics
    # --------------------------------------------------------

    local_mean = uniform_filter(
        img,
        size=window_size,
        mode="reflect"
    )

    local_sq_mean = uniform_filter(
        img ** 2,
        size=window_size,
        mode="reflect"
    )

    local_var = np.maximum(
        local_sq_mean
        -
        local_mean ** 2,
        0.0
    )

    local_std = np.sqrt(
        local_var
    )

    # --------------------------------------------------------
    # Pixel-by-pixel sigma-range selection
    # --------------------------------------------------------

    for y in range(H):

        for x in range(W):

            window = padded[
                y:y + window_size,
                x:x + window_size
            ]

            mu = local_mean[y, x]
            std = local_std[y, x]

            lower = max(
                mu - sigma_factor * std,
                eps
            )

            upper = (
                mu
                +
                sigma_factor * std
            )

            selected = window[
                (window >= lower)
                &
                (window <= upper)
            ]

            # ------------------------------------------------
            # Fallback if too few pixels survive
            # ------------------------------------------------

            if selected.size < 5:
                selected = window.reshape(-1)

            selected_mean = selected.mean()

            selected_var = selected.var()

            # ------------------------------------------------
            # Adaptive Lee/MMSE weight
            # ------------------------------------------------

            noise_var = max(
                selected_var,
                eps
            )

            total_var = max(
                local_var[y, x],
                eps
            )

            weight = (
                total_var - noise_var
            ) / total_var

            weight = np.clip(
                weight,
                0.0,
                1.0
            )

            output[y, x] = (
                selected_mean
                +
                weight
                *
                (
                    img[y, x]
                    -
                    selected_mean
                )
            )

    return np.maximum(
        output,
        eps
    ).astype(np.float32)


# ============================================================
# APPLY TO SAME REFERENCE SCENES
# ============================================================

lee_sigma_records = []

lee_sigma_visual_records = []


for _, patch_row in homogeneous_patches.iterrows():

    scene_id_i = patch_row["scene_id"]

    manifest_row = train_df[
        train_df["scene_id"] == scene_id_i
    ].iloc[0]

    path = locate_input_file(
        manifest_row["sar_raw_path"]
    )

    with rasterio.open(path) as src:

        raw_db = src.read().astype(
            np.float32
        )

    valid = (
        np.isfinite(raw_db[0])
        &
        np.isfinite(raw_db[1])
    )

    assert valid.all()

    filtered_db = np.empty_like(
        raw_db
    )


    for band_idx in range(2):

        linear = db_to_linear(
            raw_db[band_idx],
            valid
        )

        filtered_linear = lee_sigma_filter(
            linear,
            window_size=7,
            sigma_factor=0.9
        )

        filtered_db[band_idx] = linear_to_db(
            filtered_linear,
            valid
        )


    # --------------------------------------------------------
    # Frozen homogeneous patch
    # --------------------------------------------------------

    x0 = int(patch_row["x"])
    y0 = int(patch_row["y"])
    ps = int(patch_row["patch_size"])

    record = {
        "scene_id": scene_id_i,
        "category": patch_row["category"]
    }


    for band_idx, band_name in enumerate(
        ["VV", "VH"]
    ):

        raw_patch_db = raw_db[
            band_idx,
            y0:y0+ps,
            x0:x0+ps
        ]

        sigma_patch_db = filtered_db[
            band_idx,
            y0:y0+ps,
            x0:x0+ps
        ]


        raw_linear = np.power(
            10.0,
            raw_patch_db / 10.0
        )

        sigma_linear = np.power(
            10.0,
            sigma_patch_db / 10.0
        )


        raw_enl = patch_statistics(
            raw_linear
        )[3]

        sigma_enl = patch_statistics(
            sigma_linear
        )[3]


        # ----------------------------------------------------
        # Edge preservation
        # ----------------------------------------------------

        edge_metrics = edge_preservation_metrics(
            raw_db[band_idx],
            filtered_db[band_idx]
        )


        prefix = band_name.lower()

        record[
            f"{prefix}_raw_enl"
        ] = raw_enl

        record[
            f"{prefix}_lee_sigma_enl"
        ] = sigma_enl

        record[
            f"{prefix}_enl_gain"
        ] = sigma_enl / raw_enl

        record[
            f"{prefix}_egr"
        ] = edge_metrics["egr"]

        record[
            f"{prefix}_egc"
        ] = edge_metrics["egc"]


    lee_sigma_records.append(
        record
    )

    lee_sigma_visual_records.append(
        (
            scene_id_i,
            patch_row["category"],
            raw_db,
            filtered_db
        )
    )


lee_sigma_results = pd.DataFrame(
    lee_sigma_records
)


# ============================================================
# RESULTS
# ============================================================

print("=" * 82)
print("P3-C10 — LEE SIGMA RESULTS")
print("=" * 82)

print(
    lee_sigma_results.to_string(
        index=False
    )
)


print("\nSUMMARY")
print("-" * 82)

print(
    "Mean VV ENL gain:",
    lee_sigma_results[
        "vv_enl_gain"
    ].mean()
)

print(
    "Mean VH ENL gain:",
    lee_sigma_results[
        "vh_enl_gain"
    ].mean()
)

print(
    "Mean VV EGR:",
    lee_sigma_results[
        "vv_egr"
    ].mean()
)

print(
    "Mean VH EGR:",
    lee_sigma_results[
        "vh_egr"
    ].mean()
)

print(
    "Mean VV EGC:",
    lee_sigma_results[
        "vv_egc"
    ].mean()
)

print(
    "Mean VH EGC:",
    lee_sigma_results[
        "vh_egc"
    ].mean()
)

print("\nLee Sigma comparison complete ✅")


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C11
# SECTION: P3-S5 — FREEZE CLASSICAL DESPECKLING RESULTS
#
# PURPOSE:
#   Save the current Raw / Refined Lee / Lee Sigma comparison
#   before moving to deep-learning despeckling.
# ============================================================

from pathlib import Path
import pandas as pd

RESULT_ROOT = (
    OUTPUT_ROOT /
    "sar_despeckling_results"
)

RESULT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Save individual detailed results
# ------------------------------------------------------------

refined_lee_results.to_csv(
    RESULT_ROOT /
    "refined_lee_enl_results.csv",
    index=False
)

refined_lee_edge_results.to_csv(
    RESULT_ROOT /
    "refined_lee_edge_results.csv",
    index=False
)

lee_sigma_results.to_csv(
    RESULT_ROOT /
    "lee_sigma_results.csv",
    index=False
)


# ------------------------------------------------------------
# Build compact summary
# ------------------------------------------------------------

refined_summary = {
    "method": "Refined Lee",

    "vv_enl_gain":
        refined_lee_results[
            "vv_enl_gain"
        ].mean(),

    "vh_enl_gain":
        refined_lee_results[
            "vh_enl_gain"
        ].mean(),

    "vv_egr":
        refined_lee_edge_results[
            "vv_egr"
        ].mean(),

    "vh_egr":
        refined_lee_edge_results[
            "vh_egr"
        ].mean(),

    "vv_egc":
        refined_lee_edge_results[
            "vv_egc"
        ].mean(),

    "vh_egc":
        refined_lee_edge_results[
            "vh_egc"
        ].mean(),
}


sigma_summary = {
    "method": "Lee Sigma",

    "vv_enl_gain":
        lee_sigma_results[
            "vv_enl_gain"
        ].mean(),

    "vh_enl_gain":
        lee_sigma_results[
            "vh_enl_gain"
        ].mean(),

    "vv_egr":
        lee_sigma_results[
            "vv_egr"
        ].mean(),

    "vh_egr":
        lee_sigma_results[
            "vh_egr"
        ].mean(),

    "vv_egc":
        lee_sigma_results[
            "vv_egc"
        ].mean(),

    "vh_egc":
        lee_sigma_results[
            "vh_egc"
        ].mean(),
}


classical_summary = pd.DataFrame(
    [
        refined_summary,
        sigma_summary
    ]
)

classical_summary.to_csv(
    RESULT_ROOT /
    "classical_despeckling_summary.csv",
    index=False
)


print("=" * 78)
print("P3-C11 — CLASSICAL SAR RESULTS FROZEN")
print("=" * 78)

print(
    classical_summary.to_string(
        index=False
    )
)

print("\nSaved to:")
print(RESULT_ROOT)

print(
    "\nCurrent classical leader "
    "(structure/noise trade-off): Lee Sigma ✅"
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C12
# SECTION: P3-S6 — SAR2SAR SETUP + PYTORCH COMPATIBILITY
#
# PURPOSE:
#   Install and verify the maintained SAR2SAR implementation.
#   Apply a narrow compatibility wrapper only while the bundled
#   trusted checkpoint is instantiated.
#
# IMPORTANT:
#   - The compatibility wrapper is active only during model load.
#   - torch.load is restored immediately afterward.
#   - No image is denoised in this cell.
# ============================================================

import sys
import subprocess
import torch

print("=" * 78)
print("P3-C12 — SAR2SAR SETUP + COMPATIBILITY")
print("=" * 78)

print("Python :", sys.version)
print("PyTorch:", torch.__version__)


# ------------------------------------------------------------
# Install package if needed
# ------------------------------------------------------------

try:
    import deepdespeckling
    print("\ndeepdespeckling already installed ✅")

except ImportError:

    print("\nInstalling deepdespeckling...")

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "deepdespeckling"
        ]
    )

    import deepdespeckling

    print("deepdespeckling installed ✅")


# ------------------------------------------------------------
# Import SAR2SAR implementation
# ------------------------------------------------------------

from deepdespeckling.sar2sar.sar2sar_denoiser import (
    Sar2SarDenoiser
)

from deepdespeckling.utils.constants import (
    PATCH_SIZE,
    STRIDE_SIZE
)

print("\nSAR2SAR imports successful ✅")
print("PATCH_SIZE :", PATCH_SIZE)
print("STRIDE_SIZE:", STRIDE_SIZE)


# ------------------------------------------------------------
# PyTorch 2.6+ compatibility for the trusted bundled checkpoint
#
# Older deepdespeckling checkpoints assume:
#     torch.load(..., weights_only=False)
#
# The wrapper is deliberately temporary so it cannot affect
# later FloodShift code.
# ------------------------------------------------------------

_original_torch_load = torch.load

def _sar2sar_compat_load(*args, **kwargs):

    kwargs.setdefault(
        "weights_only",
        False
    )

    return _original_torch_load(
        *args,
        **kwargs
    )


torch.load = _sar2sar_compat_load

try:

    denoiser = Sar2SarDenoiser()

finally:

    torch.load = _original_torch_load


print("\nSAR2SAR denoiser instantiated ✅")
print("torch.load restored ✅")

print("\nP3-C12 COMPLETE")


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C13
# SECTION: P3-S7 — SAR2SAR FIRST EVALUATION
#
# PURPOSE:
#   Apply pretrained SAR2SAR to the SAME three reference scenes
#   used for Refined Lee and Lee Sigma.
#
#   Evaluate:
#       - ENL gain
#       - Edge Gradient Retention (EGR)
#       - Edge Gradient Correlation (EGC)
#       - Runtime
#
# IMPORTANT:
#   SAR2SAR expects AMPLITUDE, not dB.
#
#   dB
#    -> linear intensity
#    -> sqrt = amplitude
#    -> SAR2SAR
#    -> denoised amplitude
#    -> square = intensity
#    -> dB
#
# PYTORCH 2.6+ NOTE:
#   deepdespeckling loads its bundled legacy SAR2SAR checkpoint
#   lazily inside denoise_image().
#
#   PyTorch >= 2.6 defaults torch.load(weights_only=True),
#   which is incompatible with this trusted bundled checkpoint.
#
#   Therefore torch.load is temporarily wrapped ONLY during
#   SAR2SAR inference and restored immediately afterward.
# ============================================================

import time
import numpy as np
import pandas as pd
import rasterio
import torch


print("=" * 82)
print("P3-C13 — SAR2SAR FIRST EVALUATION")
print("=" * 82)

# ------------------------------------------------------------
# Check execution device
# ------------------------------------------------------------

print("SAR2SAR device:", denoiser.device)


# ============================================================
# PYTORCH 2.6+ SAR2SAR COMPATIBILITY HELPER
# ============================================================

def sar2sar_denoise_with_compat(
    denoiser,
    amplitude,
    patch_size,
    stride_size
):
    """
    Run SAR2SAR while temporarily restoring the legacy
    torch.load(weights_only=False) behaviour required by the
    trusted checkpoint bundled with deepdespeckling.

    torch.load is ALWAYS restored afterward.
    """

    original_torch_load = torch.load


    def compatible_torch_load(
        *args,
        **kwargs
    ):

        kwargs["weights_only"] = False

        return original_torch_load(
            *args,
            **kwargs
        )


    torch.load = compatible_torch_load


    try:

        output = denoiser.denoise_image(
            amplitude,
            patch_size=patch_size,
            stride_size=stride_size
        )


    finally:

        # ----------------------------------------------------
        # CRITICAL:
        # Never leave global torch.load patched.
        # ----------------------------------------------------

        torch.load = original_torch_load


    return output


print(
    "PyTorch 2.6+ SAR2SAR compatibility helper ready ✅"
)


# ============================================================
# STORAGE
# ============================================================

sar2sar_records = []

sar2sar_visual_records = []


# ============================================================
# PROCESS SAME FROZEN REFERENCE SCENES
# ============================================================

for _, patch_row in homogeneous_patches.iterrows():

    scene_id_i = patch_row[
        "scene_id"
    ]


    manifest_row = train_df[
        train_df["scene_id"]
        ==
        scene_id_i
    ].iloc[0]


    sar_path_i = locate_input_file(
        manifest_row[
            "sar_raw_path"
        ]
    )


    with rasterio.open(
        sar_path_i
    ) as src:

        raw_db = src.read().astype(
            np.float32
        )


    # --------------------------------------------------------
    # These reference scenes were intentionally selected from
    # scenes with complete SAR coverage.
    # --------------------------------------------------------

    valid = (
        np.isfinite(
            raw_db[0]
        )
        &
        np.isfinite(
            raw_db[1]
        )
    )


    assert valid.all(), (
        f"{scene_id_i} unexpectedly contains invalid SAR."
    )


    filtered_db = np.empty_like(
        raw_db,
        dtype=np.float32
    )


    runtimes = []


    # ========================================================
    # PROCESS VV AND VH SEPARATELY
    # ========================================================

    for band_idx, band_name in enumerate(
        [
            "VV",
            "VH"
        ]
    ):

        print(
            f"\nProcessing "
            f"{scene_id_i} — {band_name}"
        )


        # ----------------------------------------------------
        # STEP 1
        # dB -> linear intensity
        # ----------------------------------------------------

        intensity = db_to_linear(
            raw_db[
                band_idx
            ],
            valid
        )


        # ----------------------------------------------------
        # STEP 2
        # intensity -> amplitude
        #
        # intensity = amplitude^2
        # ----------------------------------------------------

        amplitude = np.sqrt(
            np.maximum(
                intensity,
                1e-12
            )
        ).astype(
            np.float32
        )


        # ----------------------------------------------------
        # STEP 3
        # SAR2SAR inference
        #
        # Compatibility wrapper is active ONLY while
        # denoise_image() loads/runs the bundled checkpoint.
        # ----------------------------------------------------

        start = time.perf_counter()


        output = sar2sar_denoise_with_compat(

            denoiser=
                denoiser,

            amplitude=
                amplitude,

            patch_size=
                PATCH_SIZE,

            stride_size=
                STRIDE_SIZE
        )


        elapsed = (
            time.perf_counter()
            -
            start
        )


        runtimes.append(
            elapsed
        )


        # ----------------------------------------------------
        # Current deepdespeckling implementation returns:
        #
        # {
        #     "noisy": ...,
        #     "denoised": ...
        # }
        # ----------------------------------------------------

        if not isinstance(
            output,
            dict
        ):

            raise TypeError(
                "Unexpected SAR2SAR output type: "
                f"{type(output)}"
            )


        if "denoised" not in output:

            raise KeyError(
                "SAR2SAR output does not contain "
                "'denoised'. Keys: "
                f"{output.keys()}"
            )


        denoised_amplitude = np.asarray(
            output[
                "denoised"
            ],
            dtype=np.float32
        )


        # ----------------------------------------------------
        # Validate output
        # ----------------------------------------------------

        assert (
            denoised_amplitude.shape
            ==
            amplitude.shape
        ), (
            f"Shape mismatch: "
            f"{denoised_amplitude.shape} vs "
            f"{amplitude.shape}"
        )


        if not np.isfinite(
            denoised_amplitude
        ).all():

            raise ValueError(
                f"SAR2SAR produced non-finite values "
                f"for {scene_id_i} {band_name}"
            )


        # ----------------------------------------------------
        # SAR amplitude must be non-negative
        # ----------------------------------------------------

        denoised_amplitude = np.maximum(
            denoised_amplitude,
            1e-12
        )


        # ----------------------------------------------------
        # STEP 4
        # amplitude -> intensity
        # ----------------------------------------------------

        denoised_intensity = (
            denoised_amplitude
            **
            2
        )


        # ----------------------------------------------------
        # STEP 5
        # intensity -> dB
        # ----------------------------------------------------

        filtered_db[
            band_idx
        ] = linear_to_db(
            denoised_intensity,
            valid
        )


        print(
            f"  Runtime: "
            f"{elapsed:.3f} sec"
        )


        print(
            "  Output amplitude range:",
            float(
                denoised_amplitude.min()
            ),
            "to",
            float(
                denoised_amplitude.max()
            )
        )


    # ========================================================
    # EVALUATE SAME FROZEN HOMOGENEOUS PATCH
    # ========================================================

    x0 = int(
        patch_row[
            "x"
        ]
    )

    y0 = int(
        patch_row[
            "y"
        ]
    )

    ps = int(
        patch_row[
            "patch_size"
        ]
    )


    record = {

        "scene_id":
            scene_id_i,

        "category":
            patch_row[
                "category"
            ],

        "runtime_sec_total":
            sum(
                runtimes
            )
    }


    for band_idx, band_name in enumerate(
        [
            "VV",
            "VH"
        ]
    ):

        prefix = (
            band_name.lower()
        )


        # ----------------------------------------------------
        # Frozen patch
        # ----------------------------------------------------

        raw_patch_db = raw_db[
            band_idx,
            y0:y0 + ps,
            x0:x0 + ps
        ]


        sar2sar_patch_db = filtered_db[
            band_idx,
            y0:y0 + ps,
            x0:x0 + ps
        ]


        # ----------------------------------------------------
        # ENL in linear-intensity domain
        # ----------------------------------------------------

        raw_patch_linear = np.power(
            10.0,
            raw_patch_db / 10.0
        )


        sar2sar_patch_linear = np.power(
            10.0,
            sar2sar_patch_db / 10.0
        )


        raw_enl = patch_statistics(
            raw_patch_linear
        )[3]


        sar2sar_enl = patch_statistics(
            sar2sar_patch_linear
        )[3]


        # ----------------------------------------------------
        # Same edge-preservation metric used for all methods
        # ----------------------------------------------------

        edge_metrics = (
            edge_preservation_metrics(

                raw_db[
                    band_idx
                ],

                filtered_db[
                    band_idx
                ]
            )
        )


        record[
            f"{prefix}_raw_enl"
        ] = raw_enl


        record[
            f"{prefix}_sar2sar_enl"
        ] = sar2sar_enl


        record[
            f"{prefix}_enl_gain"
        ] = (
            sar2sar_enl
            /
            raw_enl
        )


        record[
            f"{prefix}_egr"
        ] = edge_metrics[
            "egr"
        ]


        record[
            f"{prefix}_egc"
        ] = edge_metrics[
            "egc"
        ]


    sar2sar_records.append(
        record
    )


    sar2sar_visual_records.append(
        (
            scene_id_i,
            patch_row[
                "category"
            ],
            raw_db,
            filtered_db
        )
    )


# ============================================================
# RESULTS TABLE
# ============================================================

sar2sar_results = pd.DataFrame(
    sar2sar_records
)


print(
    "\n"
    +
    "=" * 82
)

print(
    "SAR2SAR RESULTS"
)

print(
    "=" * 82
)


print(
    sar2sar_results.to_string(
        index=False
    )
)


# ============================================================
# SUMMARY
# ============================================================

print(
    "\nSUMMARY"
)

print(
    "-" * 82
)


print(
    "Mean VV ENL gain:",
    sar2sar_results[
        "vv_enl_gain"
    ].mean()
)


print(
    "Mean VH ENL gain:",
    sar2sar_results[
        "vh_enl_gain"
    ].mean()
)


print(
    "Mean VV EGR:",
    sar2sar_results[
        "vv_egr"
    ].mean()
)


print(
    "Mean VH EGR:",
    sar2sar_results[
        "vh_egr"
    ].mean()
)


print(
    "Mean VV EGC:",
    sar2sar_results[
        "vv_egc"
    ].mean()
)


print(
    "Mean VH EGC:",
    sar2sar_results[
        "vh_egc"
    ].mean()
)


print(
    "Mean runtime per 2-band scene:",
    sar2sar_results[
        "runtime_sec_total"
    ].mean(),
    "sec"
)


# ============================================================
# FINAL SAFETY CHECK
# ============================================================

assert len(
    sar2sar_results
) == len(
    homogeneous_patches
)


assert np.isfinite(
    sar2sar_results.select_dtypes(
        include=[
            np.number
        ]
    ).to_numpy()
).all()


print(
    "\ntorch.load currently restored:",
    torch.load.__name__
    !=
    "compatible_torch_load"
)


print(
    "\nSAR2SAR first evaluation complete ✅"
)

In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C14
# SECTION: P3-S6 — HOMOMORPHIC BM3D SETUP
#
# PURPOSE:
#   Install and verify the modern reference BM3D package.
#
# IMPORTANT:
#   This is NOT labelled "SAR-BM3D".
#   We will use BM3D in a homomorphic/log-domain SAR pipeline.
# ============================================================

import sys
import subprocess

print("=" * 72)
print("P3-C14 — HOMOMORPHIC BM3D SETUP")
print("=" * 72)

try:
    import bm3d

    print("bm3d already installed ✅")

except ImportError:

    print("Installing bm3d...")

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "bm3d"
        ]
    )

    import bm3d

    print("bm3d installed ✅")


print("\nBM3D module:", bm3d)

print(
    "\nHomomorphic BM3D environment ready ✅"
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C15
# SECTION: P3-S6 — TRAIN-ONLY BM3D NOISE ESTIMATION
#
# PURPOSE:
#   Estimate a robust log-domain noise scale from the frozen
#   homogeneous TRAIN patches.
#
# NO filtering is performed yet.
# ============================================================

import numpy as np
import pandas as pd
import rasterio

noise_records = []

for _, patch_row in homogeneous_patches.iterrows():

    scene_id_i = patch_row["scene_id"]

    manifest_row = train_df[
        train_df["scene_id"] == scene_id_i
    ].iloc[0]

    sar_path_i = locate_input_file(
        manifest_row["sar_raw_path"]
    )

    with rasterio.open(sar_path_i) as src:
        raw_db = src.read().astype(np.float32)

    x0 = int(patch_row["x"])
    y0 = int(patch_row["y"])
    ps = int(patch_row["patch_size"])

    for band_idx, band_name in enumerate(
        ["VV", "VH"]
    ):

        patch_db = raw_db[
            band_idx,
            y0:y0 + ps,
            x0:x0 + ps
        ]

        # --------------------------------------------
        # dB -> linear intensity
        # --------------------------------------------

        patch_linear = np.power(
            10.0,
            patch_db / 10.0
        )

        # --------------------------------------------
        # log-intensity domain
        # --------------------------------------------

        log_patch = np.log(
            np.maximum(
                patch_linear,
                1e-12
            )
        )

        # Robust scale estimate using MAD
        median = np.median(log_patch)

        mad = np.median(
            np.abs(
                log_patch - median
            )
        )

        sigma_mad = (
            1.4826 * mad
        )

        noise_records.append({
            "scene_id": scene_id_i,
            "category": patch_row["category"],
            "band": band_name,
            "log_sigma_mad": float(sigma_mad),
            "log_std": float(log_patch.std())
        })


bm3d_noise_stats = pd.DataFrame(
    noise_records
)


print("=" * 76)
print("P3-C15 — LOG-SPECKLE SCALE ESTIMATION")
print("=" * 76)

print(
    bm3d_noise_stats.to_string(
        index=False
    )
)


print("\nSUMMARY")
print("-" * 76)

print(
    bm3d_noise_stats.groupby("band")[
        ["log_sigma_mad", "log_std"]
    ].median()
)


BM3D_SIGMA_VV = (
    bm3d_noise_stats[
        bm3d_noise_stats["band"] == "VV"
    ]["log_sigma_mad"].median()
)

BM3D_SIGMA_VH = (
    bm3d_noise_stats[
        bm3d_noise_stats["band"] == "VH"
    ]["log_sigma_mad"].median()
)


print("\nSelected TRAIN-only robust estimates:")

print(
    f"VV sigma = {BM3D_SIGMA_VV:.6f}"
)

print(
    f"VH sigma = {BM3D_SIGMA_VH:.6f}"
)

print(
    "\nNoise estimation complete ✅"
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C16
# SECTION: P3-S6 — HOMOMORPHIC BM3D
#
# PURPOSE:
#   Apply BM3D in log-intensity space to the SAME three
#   reference scenes.
#
# PIPELINE:
#   dB
#    -> linear intensity
#    -> natural log
#    -> BM3D
#    -> exp
#    -> radiometric mean correction
#    -> dB
#
# EVALUATION:
#   ENL
#   EGR
#   EGC
#   runtime
#
# IMPORTANT:
#   This is called "Homomorphic BM3D", NOT "SAR-BM3D".
# ============================================================

import time
import numpy as np
import pandas as pd
import rasterio
import bm3d


print("=" * 82)
print("P3-C16 — HOMOMORPHIC BM3D EVALUATION")
print("=" * 82)

BM3D_SIGMAS = {
    "VV": float(BM3D_SIGMA_VV),
    "VH": float(BM3D_SIGMA_VH),
}

print("\nScreening log-domain sigma values:")
print(BM3D_SIGMAS)


bm3d_records = []
bm3d_visual_records = []


# ============================================================
# PROCESS SAME THREE REFERENCE SCENES
# ============================================================

for _, patch_row in homogeneous_patches.iterrows():

    scene_id_i = patch_row["scene_id"]

    manifest_row = train_df[
        train_df["scene_id"] == scene_id_i
    ].iloc[0]

    sar_path_i = locate_input_file(
        manifest_row["sar_raw_path"]
    )

    with rasterio.open(sar_path_i) as src:
        raw_db = src.read().astype(np.float32)


    valid = (
        np.isfinite(raw_db[0])
        &
        np.isfinite(raw_db[1])
    )

    # These three reference scenes were chosen from
    # complete-coverage scenes.
    assert valid.all()


    filtered_db = np.empty_like(
        raw_db,
        dtype=np.float32
    )

    runtimes = []


    # ========================================================
    # VV / VH separately
    # ========================================================

    for band_idx, band_name in enumerate(
        ["VV", "VH"]
    ):

        print(
            f"\nProcessing {scene_id_i} — {band_name}"
        )

        sigma = BM3D_SIGMAS[
            band_name
        ]


        # ----------------------------------------------------
        # STEP 1: dB -> linear intensity
        # ----------------------------------------------------

        intensity = db_to_linear(
            raw_db[band_idx],
            valid
        ).astype(np.float64)


        # ----------------------------------------------------
        # STEP 2: intensity -> log intensity
        # ----------------------------------------------------

        log_intensity = np.log(
            np.maximum(
                intensity,
                1e-12
            )
        )


        # ----------------------------------------------------
        # STEP 3: BM3D in log domain
        # ----------------------------------------------------

        start = time.perf_counter()

        denoised_log = bm3d.bm3d(
            log_intensity,
            sigma_psd=sigma,
            stage_arg=bm3d.BM3DStages.ALL_STAGES
        )

        elapsed = (
            time.perf_counter()
            -
            start
        )

        runtimes.append(
            elapsed
        )


        denoised_log = np.asarray(
            denoised_log,
            dtype=np.float64
        )


        # ----------------------------------------------------
        # STEP 4: log -> linear intensity
        # ----------------------------------------------------

        denoised_intensity = np.exp(
            denoised_log
        )


        # ----------------------------------------------------
        # STEP 5: preserve scene-level mean backscatter
        #
        # Homomorphic processing can introduce a multiplicative
        # radiometric bias after exponentiation.
        # ----------------------------------------------------

        raw_mean = float(
            intensity.mean()
        )

        den_mean = float(
            denoised_intensity.mean()
        )

        radiometric_scale = (
            raw_mean
            /
            max(den_mean, 1e-12)
        )

        denoised_intensity *= (
            radiometric_scale
        )


        # ----------------------------------------------------
        # STEP 6: linear -> dB
        # ----------------------------------------------------

        filtered_db[
            band_idx
        ] = linear_to_db(
            denoised_intensity,
            valid
        )


        print(
            f"  sigma             : {sigma:.6f}"
        )

        print(
            f"  runtime           : {elapsed:.3f} sec"
        )

        print(
            f"  radiometric scale : {radiometric_scale:.6f}"
        )


    # ========================================================
    # SAME FROZEN PATCH EVALUATION
    # ========================================================

    x0 = int(
        patch_row["x"]
    )

    y0 = int(
        patch_row["y"]
    )

    ps = int(
        patch_row["patch_size"]
    )


    record = {
        "scene_id": scene_id_i,
        "category": patch_row["category"],
        "runtime_sec_total": sum(runtimes)
    }


    for band_idx, band_name in enumerate(
        ["VV", "VH"]
    ):

        prefix = band_name.lower()


        raw_patch_db = raw_db[
            band_idx,
            y0:y0 + ps,
            x0:x0 + ps
        ]

        bm3d_patch_db = filtered_db[
            band_idx,
            y0:y0 + ps,
            x0:x0 + ps
        ]


        # ----------------------------------------------------
        # ENL in linear intensity
        # ----------------------------------------------------

        raw_patch_linear = np.power(
            10.0,
            raw_patch_db / 10.0
        )

        bm3d_patch_linear = np.power(
            10.0,
            bm3d_patch_db / 10.0
        )


        raw_enl = patch_statistics(
            raw_patch_linear
        )[3]

        bm3d_enl = patch_statistics(
            bm3d_patch_linear
        )[3]


        # ----------------------------------------------------
        # Same edge metrics
        # ----------------------------------------------------

        edge_metrics = edge_preservation_metrics(
            raw_db[band_idx],
            filtered_db[band_idx]
        )


        record[
            f"{prefix}_raw_enl"
        ] = raw_enl

        record[
            f"{prefix}_bm3d_enl"
        ] = bm3d_enl

        record[
            f"{prefix}_enl_gain"
        ] = (
            bm3d_enl / raw_enl
        )

        record[
            f"{prefix}_egr"
        ] = edge_metrics["egr"]

        record[
            f"{prefix}_egc"
        ] = edge_metrics["egc"]


    bm3d_records.append(
        record
    )

    bm3d_visual_records.append(
        (
            scene_id_i,
            patch_row["category"],
            raw_db,
            filtered_db
        )
    )


# ============================================================
# RESULTS
# ============================================================

bm3d_results = pd.DataFrame(
    bm3d_records
)


print("\n" + "=" * 82)
print("HOMOMORPHIC BM3D RESULTS")
print("=" * 82)

print(
    bm3d_results.to_string(
        index=False
    )
)


print("\nSUMMARY")
print("-" * 82)

print(
    "Mean VV ENL gain:",
    bm3d_results[
        "vv_enl_gain"
    ].mean()
)

print(
    "Mean VH ENL gain:",
    bm3d_results[
        "vh_enl_gain"
    ].mean()
)

print(
    "Mean VV EGR:",
    bm3d_results[
        "vv_egr"
    ].mean()
)

print(
    "Mean VH EGR:",
    bm3d_results[
        "vh_egr"
    ].mean()
)

print(
    "Mean VV EGC:",
    bm3d_results[
        "vv_egc"
    ].mean()
)

print(
    "Mean VH EGC:",
    bm3d_results[
        "vh_egc"
    ].mean()
)

print(
    "Mean runtime per 2-band scene:",
    bm3d_results[
        "runtime_sec_total"
    ].mean(),
    "sec"
)

print(
    "\nHomomorphic BM3D evaluation complete ✅"
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C17
# SECTION: P3-S6 — FREEZE EXISTING SAR DESPECKLING SCREENING
#
# PURPOSE:
#   Build one canonical comparison table containing:
#       Refined Lee
#       Lee Sigma
#       SAR2SAR
#       Homomorphic BM3D
#
# IMPORTANT:
#   This screening does NOT select the final preprocessing
#   winner. Downstream segmentation will make that decision.
# ============================================================

import pandas as pd

print("=" * 90)
print("P3-C17 — EXISTING SAR DESPECKLING METHODS")
print("=" * 90)


# ------------------------------------------------------------
# SAR2SAR summary
# ------------------------------------------------------------

sar2sar_summary = {

    "method":
        "SAR2SAR",

    "vv_enl_gain":
        sar2sar_results[
            "vv_enl_gain"
        ].mean(),

    "vh_enl_gain":
        sar2sar_results[
            "vh_enl_gain"
        ].mean(),

    "vv_egr":
        sar2sar_results[
            "vv_egr"
        ].mean(),

    "vh_egr":
        sar2sar_results[
            "vh_egr"
        ].mean(),

    "vv_egc":
        sar2sar_results[
            "vv_egc"
        ].mean(),

    "vh_egc":
        sar2sar_results[
            "vh_egc"
        ].mean(),

    "runtime_sec_per_scene":
        sar2sar_results[
            "runtime_sec_total"
        ].mean()
}


# ------------------------------------------------------------
# Homomorphic BM3D summary
# ------------------------------------------------------------

bm3d_summary = {

    "method":
        "Homomorphic BM3D",

    "vv_enl_gain":
        bm3d_results[
            "vv_enl_gain"
        ].mean(),

    "vh_enl_gain":
        bm3d_results[
            "vh_enl_gain"
        ].mean(),

    "vv_egr":
        bm3d_results[
            "vv_egr"
        ].mean(),

    "vh_egr":
        bm3d_results[
            "vh_egr"
        ].mean(),

    "vv_egc":
        bm3d_results[
            "vv_egc"
        ].mean(),

    "vh_egc":
        bm3d_results[
            "vh_egc"
        ].mean(),

    "runtime_sec_per_scene":
        bm3d_results[
            "runtime_sec_total"
        ].mean()
}


# ------------------------------------------------------------
# Classical table + deep/non-local comparators
# ------------------------------------------------------------

classical_extended = classical_summary.copy()

classical_extended[
    "runtime_sec_per_scene"
] = pd.NA


existing_methods_summary = pd.concat(
    [
        classical_extended,
        pd.DataFrame(
            [
                sar2sar_summary,
                bm3d_summary
            ]
        )
    ],
    ignore_index=True
)


print(
    existing_methods_summary.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# Save canonical screening outputs
# ------------------------------------------------------------

existing_methods_path = (
    RESULT_ROOT /
    "sar_existing_methods_comparison_v2.csv"
)

existing_methods_summary.to_csv(
    existing_methods_path,
    index=False
)

sar2sar_results.to_csv(
    RESULT_ROOT /
    "sar2sar_detailed_results.csv",
    index=False
)

bm3d_results.to_csv(
    RESULT_ROOT /
    "homomorphic_bm3d_detailed_results.csv",
    index=False
)


print("\nSaved:")
print(existing_methods_path)

print(
    "\nExisting-method screening frozen ✅"
)

print(
    "No final despeckling winner selected yet."
)


## 3. Proposed DP-SSD self-supervised preprocessing


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C18
# SECTION: P3-S8 — SELF-SUPERVISED SAMPLE CONSTRUCTION
#
# PURPOSE:
#   Construct ONE valid dual-polarization self-supervised
#   training sample before building any neural network.
#
# PIPELINE:
#   SAR dB
#      ↓
#   linear intensity
#      ↓
#   natural log
#      ↓
#   2×2 interleaved sub-images
#      ↓
#   shared VV/VH masking
#      ↓
#   masked input + original masked targets
#
# IMPORTANT:
#   - TRAIN data only
#   - NO segmentation labels
#   - NO clean SAR targets
# ============================================================

import numpy as np
import rasterio
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

MASK_RATIO = 0.10
RANDOM_SEED = 42

rng = np.random.default_rng(
    RANDOM_SEED
)


# ------------------------------------------------------------
# Select a fully valid TRAIN scene automatically
# ------------------------------------------------------------

sample_scene_id = (
    homogeneous_patches
    .iloc[0]["scene_id"]
)

manifest_row = train_df[
    train_df["scene_id"] == sample_scene_id
].iloc[0]

sar_path_i = locate_input_file(
    manifest_row["sar_raw_path"]
)


with rasterio.open(sar_path_i) as src:

    raw_db = src.read().astype(
        np.float32
    )


print("=" * 78)
print("P3-C18 — SELF-SUPERVISED SAMPLE CONSTRUCTION")
print("=" * 78)

print("Scene:", sample_scene_id)
print("Raw shape:", raw_db.shape)


# ------------------------------------------------------------
# Validity
# ------------------------------------------------------------

valid = (
    np.isfinite(raw_db[0])
    &
    np.isfinite(raw_db[1])
)

assert valid.all(), (
    "Reference scene should have complete SAR coverage."
)


# ============================================================
# STEP 1 — dB → linear intensity
# ============================================================

linear = np.empty_like(
    raw_db,
    dtype=np.float32
)

for band_idx in range(2):

    linear[band_idx] = db_to_linear(
        raw_db[band_idx],
        valid
    )


# ============================================================
# STEP 2 — linear intensity → natural log intensity
# ============================================================

log_sar = np.log(
    np.maximum(
        linear,
        1e-12
    )
).astype(np.float32)


print(
    "\nLog-SAR shape:",
    log_sar.shape
)


# ============================================================
# STEP 3 — construct four interleaved sub-images
#
# Original:
#
# A B A B
# C D C D
# A B A B
# C D C D
#
# gives:
#
# A = even/even
# B = even/odd
# C = odd/even
# D = odd/odd
#
# Each output is 256 × 256.
# ============================================================

def make_interleaved_views(x):

    views = {}

    offsets = {
        "EE": (0, 0),
        "EO": (0, 1),
        "OE": (1, 0),
        "OO": (1, 1),
    }

    for name, (row_offset, col_offset) in offsets.items():

        views[name] = x[
            :,
            row_offset::2,
            col_offset::2
        ]

    return views


views = make_interleaved_views(
    log_sar
)


print("\nInterleaved views:")

for name, view in views.items():

    print(
        f"  {name}: {view.shape}"
    )


# ------------------------------------------------------------
# Use EE only for this demonstration.
#
# During training ALL four views will be used.
# ------------------------------------------------------------

original = views["EE"].copy()

C, H, W = original.shape

assert C == 2


# ============================================================
# STEP 4 — create ONE shared masking map
#
# The SAME pixels are hidden in VV and VH.
# ============================================================

mask = (
    rng.random(
        (H, W)
    )
    <
    MASK_RATIO
)


print(
    "\nMasked spatial positions:",
    f"{mask.sum():,}"
)

print(
    "Actual mask percentage:",
    f"{100 * mask.mean():.2f}%"
)


# ============================================================
# STEP 5 — replace masked pixels with neighboring observations
#
# We must NOT leave the original target visible.
#
# Each masked position receives a value from one of:
#
#   ↑ ↓ ← →
#
# The replacement direction is random.
# ============================================================

def neighbour_replacement(
    image,
    mask,
    rng
):
    """
    image:
        [C, H, W]

    mask:
        [H, W]

    Returns:
        masked copy of image.
    """

    C, H, W = image.shape

    masked_image = image.copy()


    # Reflect padding prevents wrap-around leakage.
    padded = np.pad(
        image,
        (
            (0, 0),
            (1, 1),
            (1, 1)
        ),
        mode="reflect"
    )


    # Four neighboring candidates
    neighbours = np.stack(
        [
            # up
            padded[
                :,
                0:H,
                1:W+1
            ],

            # down
            padded[
                :,
                2:H+2,
                1:W+1
            ],

            # left
            padded[
                :,
                1:H+1,
                0:W
            ],

            # right
            padded[
                :,
                1:H+1,
                2:W+2
            ],

        ],
        axis=0
    )

    # One random neighbour direction for every spatial position
    direction_map = rng.integers(
        0,
        4,
        size=(H, W)
    )


    yy, xx = np.indices(
        (H, W)
    )


    for channel in range(C):

        replacements = neighbours[
            direction_map,
            channel,
            yy,
            xx
        ]

        masked_image[
            channel,
            mask
        ] = replacements[
            mask
        ]


    return masked_image


masked_input = neighbour_replacement(
    original,
    mask,
    rng
)


# ============================================================
# STEP 6 — target
#
# Target values are simply the ORIGINAL observations at the
# locations we intentionally hid.
#
# Loss will later be calculated ONLY on these positions.
# ============================================================

target = original.copy()


# ============================================================
# SANITY CHECKS
# ============================================================

assert masked_input.shape == original.shape

assert target.shape == original.shape

assert mask.shape == (
    original.shape[1],
    original.shape[2]
)

# Both polarizations use exactly the same spatial mask.
assert mask.dtype == bool


# Unmasked pixels must remain unchanged.
assert np.allclose(
    masked_input[:, ~mask],
    original[:, ~mask]
)


print("\nTensor structure:")

print(
    "Original     :",
    original.shape
)

print(
    "Masked input :",
    masked_input.shape
)

print(
    "Target       :",
    target.shape
)

print(
    "Mask         :",
    mask.shape
)


# ============================================================
# VISUAL CHECK
# ============================================================

fig, axes = plt.subplots(
    2,
    3,
    figsize=(14, 8)
)


band_names = [
    "VV",
    "VH"
]


for band_idx, band_name in enumerate(
    band_names
):

    original_band = original[
        band_idx
    ]

    masked_band = masked_input[
        band_idx
    ]


    lo, hi = np.percentile(
        original_band,
        [2, 98]
    )


    axes[
        band_idx,
        0
    ].imshow(
        original_band,
        cmap="gray",
        vmin=lo,
        vmax=hi
    )

    axes[
        band_idx,
        0
    ].set_title(
        f"{band_name} — original log-intensity"
    )


    axes[
        band_idx,
        1
    ].imshow(
        masked_band,
        cmap="gray",
        vmin=lo,
        vmax=hi
    )

    axes[
        band_idx,
        1
    ].set_title(
        f"{band_name} — masked input"
    )


    axes[
        band_idx,
        2
    ].imshow(
        mask,
        cmap="gray"
    )

    axes[
        band_idx,
        2
    ].set_title(
        "Shared masked locations"
    )


    for j in range(3):

        axes[
            band_idx,
            j
        ].axis("off")


plt.tight_layout()

plt.show()


print(
    "\nSelf-supervised sample construction verified ✅"
)

print(
    "No clean SAR target or segmentation label was used."
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C19
# SECTION: P3-S8 — DP-SSD VALID-CROP AUDIT
#
# PURPOSE:
#   Determine which TRAIN scenes / interleaved views contain
#   at least one fully valid 128x128 crop.
#
# WHY:
#   DP-SSD should never learn from fabricated values inserted
#   into SAR no-data regions.
#
# IMPORTANT:
#   - TRAIN only
#   - SAR validity only
#   - no labels
#   - no validation/test
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import rasterio


# ------------------------------------------------------------
# Canonical DP-SSD artifact directory
# ------------------------------------------------------------

DPSSD_ROOT = (
    OUTPUT_ROOT /
    "dpssd"
)

DPSSD_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

CROP_SIZE = 128

VIEW_OFFSETS = {
    "EE": (0, 0),
    "EO": (0, 1),
    "OE": (1, 0),
    "OO": (1, 1),
}


# ------------------------------------------------------------
# Integral-image helper
# ------------------------------------------------------------

def valid_crop_locations(valid_mask, crop_size):
    """
    Return top-left coordinates of all crop_size x crop_size
    windows that are 100% valid.

    valid_mask:
        [H, W] boolean
    """

    x = valid_mask.astype(np.int32)

    # Integral image with zero border
    integral = np.pad(
        x,
        ((1, 0), (1, 0)),
        mode="constant"
    ).cumsum(0).cumsum(1)

    window_sum = (
        integral[crop_size:, crop_size:]
        - integral[:-crop_size, crop_size:]
        - integral[crop_size:, :-crop_size]
        + integral[:-crop_size, :-crop_size]
    )

    required = crop_size * crop_size

    yy, xx = np.where(
        window_sum == required
    )

    return yy, xx


# ============================================================
# AUDIT ALL TRAIN SCENES
# ============================================================

crop_audit_records = []

for scene_num, (_, row) in enumerate(
    train_df.iterrows(),
    start=1
):

    sar_path_i = locate_input_file(
        row["sar_raw_path"]
    )

    with rasterio.open(sar_path_i) as src:

        x = src.read().astype(
            np.float32
        )

    valid_full = (
        np.isfinite(x[0])
        &
        np.isfinite(x[1])
    )

    scene_has_any_valid_crop = False

    for view_name, (
        row_offset,
        col_offset
    ) in VIEW_OFFSETS.items():

        valid_view = valid_full[
            row_offset::2,
            col_offset::2
        ]

        yy, xx = valid_crop_locations(
            valid_view,
            CROP_SIZE
        )

        n_valid_crops = len(yy)

        if n_valid_crops > 0:
            scene_has_any_valid_crop = True

        crop_audit_records.append({
            "scene_id": row["scene_id"],
            "region": row["region"],
            "view": view_name,

            "view_height": valid_view.shape[0],
            "view_width": valid_view.shape[1],

            "valid_pixels_view": int(
                valid_view.sum()
            ),

            "valid_pct_view": float(
                100.0 * valid_view.mean()
            ),

            "valid_crop_count": int(
                n_valid_crops
            ),

            "eligible": bool(
                n_valid_crops > 0
            ),
        })


    if (
        scene_num % 50 == 0
        or scene_num == len(train_df)
    ):
        print(
            f"Processed "
            f"{scene_num:>3}/{len(train_df)} scenes"
        )


# ============================================================
# RESULTS
# ============================================================

dpssd_crop_audit = pd.DataFrame(
    crop_audit_records
)

eligible_pairs = dpssd_crop_audit[
    dpssd_crop_audit["eligible"]
].copy()


eligible_scene_ids = set(
    eligible_pairs["scene_id"]
)

ineligible_scene_ids = (
    set(train_df["scene_id"])
    -
    eligible_scene_ids
)


print("\n" + "=" * 78)
print("P3-C19 — DP-SSD VALID-CROP AUDIT")
print("=" * 78)

print(
    "TRAIN scenes                  :",
    len(train_df)
)

print(
    "Possible scene-view pairs     :",
    len(train_df) * 4
)

print(
    "Eligible scene-view pairs     :",
    len(eligible_pairs)
)

print(
    "Eligible TRAIN scenes         :",
    len(eligible_scene_ids)
)

print(
    "Scenes with no valid 128 crop :",
    len(ineligible_scene_ids)
)


print("\nEligibility by interleaved view:")
print(
    dpssd_crop_audit.groupby(
        "view"
    )["eligible"].agg(
        ["sum", "count"]
    )
)


print("\nValid crop counts for eligible pairs:")

print(
    eligible_pairs[
        "valid_crop_count"
    ].describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)


if ineligible_scene_ids:

    print("\nScenes with no fully valid crop:")

    for scene_id_i in sorted(
        ineligible_scene_ids
    ):

        print(
            " ",
            scene_id_i
        )


# ============================================================
# SAVE
# ============================================================

crop_audit_path = (
    DPSSD_ROOT /
    "dpssd_valid_crop_audit_v1.csv"
)

eligible_path = (
    DPSSD_ROOT /
    "dpssd_eligible_scene_views_v1.csv"
)


dpssd_crop_audit.to_csv(
    crop_audit_path,
    index=False
)

eligible_pairs.to_csv(
    eligible_path,
    index=False
)


print("\nSaved:")
print(crop_audit_path)
print(eligible_path)

print(
    "\nDP-SSD crop eligibility audit complete ✅"
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C20
# SECTION: P3-S8 — DP-SSD INTERNAL TRAIN/VAL SPLIT
#
# PURPOSE:
#   Split the 308 eligible TRAIN scenes into:
#
#       DP-SSD train
#       DP-SSD internal validation
#
# IMPORTANT:
#   - Both subsets come ONLY from canonical TRAIN.
#   - Split is scene-level, never view-level.
#   - All four views of a scene stay together.
#   - Main FloodShift validation/test remain untouched.
# ============================================================

import numpy as np
import pandas as pd

DPSSD_VAL_FRACTION = 0.10
DPSSD_SPLIT_SEED = 2026

rng_split = np.random.default_rng(
    DPSSD_SPLIT_SEED
)


# ------------------------------------------------------------
# One row per eligible scene
# ------------------------------------------------------------

eligible_scene_table = (
    eligible_pairs[
        ["scene_id", "region"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

assert (
    len(eligible_scene_table)
    ==
    308
)


# ------------------------------------------------------------
# Region-aware scene split
# ------------------------------------------------------------

train_scene_ids = []
val_scene_ids = []

for region, group in eligible_scene_table.groupby(
    "region"
):

    scene_ids = group[
        "scene_id"
    ].tolist()

    rng_split.shuffle(
        scene_ids
    )

    # At least 1 internal-validation scene per region
    # whenever the region contains enough eligible scenes.
    n_val = max(
        1,
        int(
            round(
                len(scene_ids)
                *
                DPSSD_VAL_FRACTION
            )
        )
    )

    # Never consume an entire region as validation.
    n_val = min(
        n_val,
        len(scene_ids) - 1
    )

    val_scene_ids.extend(
        scene_ids[:n_val]
    )

    train_scene_ids.extend(
        scene_ids[n_val:]
    )


train_scene_ids = set(
    train_scene_ids
)

val_scene_ids = set(
    val_scene_ids
)


# ------------------------------------------------------------
# Safety
# ------------------------------------------------------------

assert train_scene_ids.isdisjoint(
    val_scene_ids
)

assert (
    train_scene_ids
    |
    val_scene_ids
) == set(
    eligible_scene_table[
        "scene_id"
    ]
)


# ------------------------------------------------------------
# Expand back to scene-view pairs
# ------------------------------------------------------------

dpssd_train_pairs = eligible_pairs[
    eligible_pairs[
        "scene_id"
    ].isin(
        train_scene_ids
    )
].reset_index(drop=True)


dpssd_val_pairs = eligible_pairs[
    eligible_pairs[
        "scene_id"
    ].isin(
        val_scene_ids
    )
].reset_index(drop=True)


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("=" * 78)
print("P3-C20 — DP-SSD INTERNAL SPLIT")
print("=" * 78)

print(
    "Eligible scenes       :",
    len(eligible_scene_table)
)

print(
    "DP-SSD train scenes   :",
    len(train_scene_ids)
)

print(
    "DP-SSD val scenes     :",
    len(val_scene_ids)
)

print(
    "\nTrain scene-view pairs:",
    len(dpssd_train_pairs)
)

print(
    "Val scene-view pairs  :",
    len(dpssd_val_pairs)
)


print("\nTRAIN regions:")

print(
    dpssd_train_pairs[
        ["scene_id", "region"]
    ]
    .drop_duplicates()[
        "region"
    ]
    .value_counts()
    .sort_index()
)


print("\nINTERNAL VAL regions:")

print(
    dpssd_val_pairs[
        ["scene_id", "region"]
    ]
    .drop_duplicates()[
        "region"
    ]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

dpssd_train_pairs.to_csv(
    DPSSD_ROOT /
    "dpssd_train_pairs_v1.csv",
    index=False
)

dpssd_val_pairs.to_csv(
    DPSSD_ROOT /
    "dpssd_internal_val_pairs_v1.csv",
    index=False
)

print(
    "\nDP-SSD internal split frozen ✅"
)

print(
    "Main FloodShift validation/test were NOT touched."
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C21
# SECTION: P3-S7 — LEAKAGE-SAFE DP-SSD NORMALIZATION
#
# PURPOSE:
#   Compute DP-SSD normalization statistics using ONLY the
#   277 DP-SSD internal-training scenes created in P3-C20.
#
# DOMAIN:
#   dB -> linear intensity -> natural log intensity
#
# IMPORTANT:
#   - DP-SSD internal validation is NOT used.
#   - Canonical FloodShift validation/test are NOT used.
#   - No segmentation labels are used.
# ============================================================

from pathlib import Path
import json
import numpy as np
import rasterio

print("=" * 78)
print("P3-C21 — LEAKAGE-SAFE DP-SSD NORMALIZATION")
print("=" * 78)


DPSSD_TRAIN_SCENE_IDS = sorted(
    train_scene_ids
)

assert len(
    DPSSD_TRAIN_SCENE_IDS
) == 277

assert set(
    DPSSD_TRAIN_SCENE_IDS
).isdisjoint(
    val_scene_ids
)


print(
    "DP-SSD training scenes:",
    len(DPSSD_TRAIN_SCENE_IDS)
)

print(
    "DP-SSD internal-val scenes:",
    len(val_scene_ids)
)


# ------------------------------------------------------------
# Exact streaming statistics
# ------------------------------------------------------------

running = {

    "VV": {
        "count": 0,
        "sum": 0.0,
        "sum_sq": 0.0
    },

    "VH": {
        "count": 0,
        "sum": 0.0,
        "sum_sq": 0.0
    }
}

band_names = [
    "VV",
    "VH"
]


for scene_num, scene_id_i in enumerate(
    DPSSD_TRAIN_SCENE_IDS,
    start=1
):

    row = train_df[
        train_df["scene_id"]
        ==
        scene_id_i
    ].iloc[0]

    sar_path_i = locate_input_file(
        row["sar_raw_path"]
    )

    with rasterio.open(
        sar_path_i
    ) as src:

        raw_db = src.read().astype(
            np.float32
        )


    valid = (
        np.isfinite(raw_db[0])
        &
        np.isfinite(raw_db[1])
    )

    if not valid.any():
        continue


    for band_idx, band_name in enumerate(
        band_names
    ):

        db_values = raw_db[
            band_idx
        ][valid].astype(
            np.float64
        )

        intensity = np.power(
            10.0,
            db_values / 10.0
        )

        log_values = np.log(
            np.maximum(
                intensity,
                1e-12
            )
        )

        running[
            band_name
        ]["count"] += log_values.size

        running[
            band_name
        ]["sum"] += float(
            log_values.sum()
        )

        running[
            band_name
        ]["sum_sq"] += float(
            np.square(
                log_values
            ).sum()
        )


    if (
        scene_num % 50 == 0
        or
        scene_num
        ==
        len(DPSSD_TRAIN_SCENE_IDS)
    ):

        print(
            f"Processed "
            f"{scene_num:>3}/"
            f"{len(DPSSD_TRAIN_SCENE_IDS)} scenes"
        )


# ------------------------------------------------------------
# Finalize statistics
# ------------------------------------------------------------

dpssd_norm_v2 = {}

for band_name in band_names:

    n = running[
        band_name
    ]["count"]

    total = running[
        band_name
    ]["sum"]

    total_sq = running[
        band_name
    ]["sum_sq"]

    mean = total / n

    variance = (
        total_sq / n
        -
        mean ** 2
    )

    std = np.sqrt(
        max(
            variance,
            0.0
        )
    )

    dpssd_norm_v2[
        band_name
    ] = {

        "valid_training_pixels":
            int(n),

        "mean_log_intensity":
            float(mean),

        "std_log_intensity":
            float(std)
    }


dpssd_norm_v2[
    "metadata"
] = {

    "version":
        "v2",

    "purpose":
        "DP-SSD self-supervised despeckler normalization",

    "scene_count":
        len(DPSSD_TRAIN_SCENE_IDS),

    "source_split":
        "DP-SSD internal training only",

    "canonical_validation_used":
        False,

    "canonical_test_used":
        False,

    "dpssd_internal_validation_used":
        False
}


# ------------------------------------------------------------
# Freeze normalization constants for later cells
# ------------------------------------------------------------

VV_MEAN = dpssd_norm_v2[
    "VV"
]["mean_log_intensity"]

VV_STD = dpssd_norm_v2[
    "VV"
]["std_log_intensity"]

VH_MEAN = dpssd_norm_v2[
    "VH"
]["mean_log_intensity"]

VH_STD = dpssd_norm_v2[
    "VH"
]["std_log_intensity"]


print("\n" + "-" * 78)
print("FINAL DP-SSD NORMALIZATION")
print("-" * 78)

print(
    f"VV mean={VV_MEAN:.6f}, "
    f"std={VV_STD:.6f}"
)

print(
    f"VH mean={VH_MEAN:.6f}, "
    f"std={VH_STD:.6f}"
)

print("\nValid training pixels:")

print(
    "VV:",
    f"{dpssd_norm_v2['VV']['valid_training_pixels']:,}"
)

print(
    "VH:",
    f"{dpssd_norm_v2['VH']['valid_training_pixels']:,}"
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

norm_v2_path = (
    DPSSD_ROOT /
    "dpssd_train_only_log_normalization_v2.json"
)

with open(
    norm_v2_path,
    "w"
) as f:

    json.dump(
        dpssd_norm_v2,
        f,
        indent=2
    )


print("\nSaved:")
print(norm_v2_path)

print(
    "\nLeakage-safe DP-SSD normalization frozen ✅"
)

print(
    "Internal validation and canonical validation/test "
    "remain unseen."
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C22
# SECTION: P3-S7 — FINAL DP-SSD TRAIN / VALIDATION LOADERS
#
# PURPOSE:
#   Build the final leakage-safe self-supervised data pipeline.
#
# TRAIN:
#   - stochastic valid crop
#   - stochastic shared VV/VH masking
#
# INTERNAL VALIDATION:
#   - fixed valid crop
#   - fixed mask
#
# IMPORTANT:
#   - normalization comes ONLY from 277 DP-SSD train scenes
#   - train/validation scenes do not overlap
#   - canonical FloodShift validation/test remain untouched
# ============================================================

import hashlib
import numpy as np
import pandas as pd
import rasterio
import torch

from torch.utils.data import Dataset, DataLoader


# ============================================================
# CONFIG
# ============================================================

DPSSD_CROP_SIZE = 128
DPSSD_MASK_RATIO = 0.10

TRAIN_BATCH_SIZE = 16
VAL_BATCH_SIZE = 16

TRAIN_SEED = 42
VAL_SEED = 2026


print("=" * 80)
print("P3-C22 — FINAL DP-SSD TRAIN / VALIDATION LOADERS")
print("=" * 80)

print("\nNormalization used:")

print(
    f"VV mean={VV_MEAN:.6f}, "
    f"std={VV_STD:.6f}"
)

print(
    f"VH mean={VH_MEAN:.6f}, "
    f"std={VH_STD:.6f}"
)


# ============================================================
# INDEX RAW SAR FILES
# ============================================================

sar_files = list(
    INPUT_ROOT.rglob(
        "*_S1Hand.tif"
    )
)

sar_file_index = {
    p.name: p
    for p in sar_files
}

print(
    "\nIndexed raw SAR files:",
    len(sar_file_index)
)

assert len(
    sar_file_index
) == 446


# ============================================================
# NORMALIZATION
# ============================================================

def normalize_log_sar(
    log_sar
):

    out = log_sar.copy().astype(
        np.float32
    )

    out[0] = (
        out[0] - VV_MEAN
    ) / VV_STD

    out[1] = (
        out[1] - VH_MEAN
    ) / VH_STD

    return out


# ============================================================
# SHARED VV/VH NEIGHBOUR MASKING
# ============================================================

def shared_neighbour_masking(
    target,
    mask_ratio,
    rng
):

    C, H, W = target.shape

    mask = (
        rng.random(
            (H, W)
        )
        <
        mask_ratio
    )

    y, x = np.where(
        mask
    )

    direction = rng.integers(
        0,
        4,
        size=len(y)
    )

    ny = y.copy()
    nx = x.copy()

    ny[
        direction == 0
    ] -= 1

    ny[
        direction == 1
    ] += 1

    nx[
        direction == 2
    ] -= 1

    nx[
        direction == 3
    ] += 1


    # Boundary correction: always use a genuine neighbour.
    bad = ny < 0
    ny[bad] = y[bad] + 1

    bad = ny >= H
    ny[bad] = y[bad] - 1

    bad = nx < 0
    nx[bad] = x[bad] + 1

    bad = nx >= W
    nx[bad] = x[bad] - 1


    masked = target.copy()

    for channel in range(C):

        masked[
            channel,
            y,
            x
        ] = target[
            channel,
            ny,
            nx
        ]


    return (
        masked.astype(
            np.float32
        ),
        mask.astype(
            bool
        )
    )


# ============================================================
# STABLE SEED FOR FIXED VALIDATION
# ============================================================

def stable_seed(
    scene_id,
    view_name,
    base_seed
):

    text = (
        f"{scene_id}|"
        f"{view_name}|"
        f"{base_seed}"
    )

    digest = hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).digest()

    return int.from_bytes(
        digest[:8],
        byteorder="little",
        signed=False
    ) % (2**32)


# ============================================================
# CREATE FIXED INTERNAL-VALIDATION PLAN
# ============================================================

val_plan_records = []

for _, row in dpssd_val_pairs.iterrows():

    scene_id_i = row[
        "scene_id"
    ]

    view_name = row[
        "view"
    ]

    filename = row.get(
        "sar_filename",
        None
    )

    if (
        filename is None
        or
        pd.isna(filename)
    ):

        manifest_row = train_df[
            train_df["scene_id"]
            ==
            scene_id_i
        ].iloc[0]

        filename = Path(
            manifest_row[
                "sar_raw_path"
            ]
        ).name


    sar_path_i = sar_file_index[
        filename
    ]

    with rasterio.open(
        sar_path_i
    ) as src:

        raw_db = src.read().astype(
            np.float32
        )


    valid_full = (
        np.isfinite(raw_db[0])
        &
        np.isfinite(raw_db[1])
    )

    ro, co = VIEW_OFFSETS[
        view_name
    ]

    valid_view = valid_full[
        ro::2,
        co::2
    ]

    yy, xx = valid_crop_locations(
        valid_view,
        DPSSD_CROP_SIZE
    )

    if len(yy) == 0:

        raise RuntimeError(
            f"No valid validation crop for "
            f"{scene_id_i}/{view_name}"
        )


    rng = np.random.default_rng(
        stable_seed(
            scene_id_i,
            view_name,
            VAL_SEED
        )
    )

    choice = int(
        rng.integers(
            0,
            len(yy)
        )
    )


    val_plan_records.append(
        {
            "scene_id":
                scene_id_i,

            "region":
                row["region"],

            "view":
                view_name,

            "sar_filename":
                filename,

            "crop_y":
                int(
                    yy[choice]
                ),

            "crop_x":
                int(
                    xx[choice]
                ),

            "mask_seed":
                int(
                    stable_seed(
                        scene_id_i,
                        view_name,
                        VAL_SEED + 100_000
                    )
                )
        }
    )


dpssd_val_plan = pd.DataFrame(
    val_plan_records
)

assert len(
    dpssd_val_plan
) == len(
    dpssd_val_pairs
)


# ============================================================
# FINAL DATASET
# ============================================================

class DPSSDFinalDataset(Dataset):

    def __init__(
        self,
        table,
        sar_file_index,
        mode="train",
        crop_size=128,
        mask_ratio=0.10,
        seed=42
    ):

        self.table = table.reset_index(
            drop=True
        )

        self.sar_file_index = (
            sar_file_index
        )

        self.mode = mode
        self.crop_size = crop_size
        self.mask_ratio = mask_ratio

        if mode not in {
            "train",
            "val"
        }:

            raise ValueError(
                "mode must be 'train' or 'val'"
            )

        # num_workers=0 below makes this reproducible within
        # a run while still evolving across training samples.
        self.rng = np.random.default_rng(
            seed
        )


    def __len__(
        self
    ):

        return len(
            self.table
        )


    def __getitem__(
        self,
        idx
    ):

        row = self.table.iloc[
            idx
        ]

        scene_id = row[
            "scene_id"
        ]

        view_name = row[
            "view"
        ]

        filename = row.get(
            "sar_filename",
            None
        )

        if (
            filename is None
            or
            pd.isna(filename)
        ):

            manifest_row = train_df[
                train_df["scene_id"]
                ==
                scene_id
            ].iloc[0]

            filename = Path(
                manifest_row[
                    "sar_raw_path"
                ]
            ).name


        sar_path = self.sar_file_index[
            filename
        ]

        with rasterio.open(
            sar_path
        ) as src:

            raw_db = src.read().astype(
                np.float32
            )


        valid_full = (
            np.isfinite(
                raw_db[0]
            )
            &
            np.isfinite(
                raw_db[1]
            )
        )


        ro, co = VIEW_OFFSETS[
            view_name
        ]

        view_db = raw_db[
            :,
            ro::2,
            co::2
        ]

        view_valid = valid_full[
            ro::2,
            co::2
        ]

        H, W = view_valid.shape
        crop = self.crop_size


        # ----------------------------------------------------
        # TRAIN: stochastic valid crop
        # ----------------------------------------------------

        if self.mode == "train":

            rng = self.rng
            found = False

            for _ in range(100):

                y = int(
                    rng.integers(
                        0,
                        H - crop + 1
                    )
                )

                x = int(
                    rng.integers(
                        0,
                        W - crop + 1
                    )
                )

                if view_valid[
                    y:y+crop,
                    x:x+crop
                ].all():

                    found = True
                    break


            if not found:

                yy, xx = valid_crop_locations(
                    view_valid,
                    crop
                )

                if len(yy) == 0:

                    raise RuntimeError(
                        f"No valid crop for "
                        f"{scene_id}/{view_name}"
                    )

                choice = int(
                    rng.integers(
                        0,
                        len(yy)
                    )
                )

                y = int(
                    yy[choice]
                )

                x = int(
                    xx[choice]
                )


        # ----------------------------------------------------
        # VAL: frozen crop and frozen mask
        # ----------------------------------------------------

        else:

            y = int(
                row[
                    "crop_y"
                ]
            )

            x = int(
                row[
                    "crop_x"
                ]
            )

            rng = np.random.default_rng(
                int(
                    row[
                        "mask_seed"
                    ]
                )
            )


        crop_db = view_db[
            :,
            y:y+crop,
            x:x+crop
        ]

        if not np.isfinite(
            crop_db
        ).all():

            raise RuntimeError(
                f"Non-finite crop: "
                f"{scene_id}/{view_name}"
            )


        # dB -> linear intensity -> log intensity
        intensity = np.power(
            10.0,
            crop_db / 10.0
        )

        log_crop = np.log(
            np.maximum(
                intensity,
                1e-12
            )
        ).astype(
            np.float32
        )


        target = normalize_log_sar(
            log_crop
        )

        masked_input, mask = (
            shared_neighbour_masking(
                target,
                self.mask_ratio,
                rng
            )
        )


        return {

            "input":
                torch.from_numpy(
                    masked_input
                ).float(),

            "target":
                torch.from_numpy(
                    target
                ).float(),

            "mask":
                torch.from_numpy(
                    mask[None]
                ).bool(),

            "scene_id":
                scene_id,

            "view":
                view_name,

            "crop_y":
                y,

            "crop_x":
                x
        }


# ============================================================
# DATASETS + SEEDED DATALOADERS
# ============================================================

dpssd_train_dataset = DPSSDFinalDataset(

    table=
        dpssd_train_pairs,

    sar_file_index=
        sar_file_index,

    mode=
        "train",

    crop_size=
        DPSSD_CROP_SIZE,

    mask_ratio=
        DPSSD_MASK_RATIO,

    seed=
        TRAIN_SEED
)


dpssd_val_dataset = DPSSDFinalDataset(

    table=
        dpssd_val_plan,

    sar_file_index=
        sar_file_index,

    mode=
        "val",

    crop_size=
        DPSSD_CROP_SIZE,

    mask_ratio=
        DPSSD_MASK_RATIO,

    seed=
        VAL_SEED
)


loader_generator = torch.Generator()

loader_generator.manual_seed(
    TRAIN_SEED
)


dpssd_train_loader = DataLoader(

    dpssd_train_dataset,

    batch_size=
        TRAIN_BATCH_SIZE,

    shuffle=True,

    num_workers=0,

    pin_memory=
        torch.cuda.is_available(),

    drop_last=True,

    generator=
        loader_generator
)


dpssd_val_loader = DataLoader(

    dpssd_val_dataset,

    batch_size=
        VAL_BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=
        torch.cuda.is_available(),

    drop_last=False
)


# ============================================================
# LEAKAGE + DETERMINISM CHECKS
# ============================================================

train_ids_final = set(
    dpssd_train_pairs[
        "scene_id"
    ]
)

val_ids_final = set(
    dpssd_val_plan[
        "scene_id"
    ]
)

assert train_ids_final.isdisjoint(
    val_ids_final
)

assert len(
    train_ids_final
) == 277

assert len(
    val_ids_final
) == 31

assert len(
    dpssd_train_dataset
) == 1108

assert len(
    dpssd_val_dataset
) == 124


val_a = dpssd_val_dataset[0]
val_b = dpssd_val_dataset[0]

assert torch.equal(
    val_a[
        "input"
    ],
    val_b[
        "input"
    ]
)

assert torch.equal(
    val_a[
        "target"
    ],
    val_b[
        "target"
    ]
)

assert torch.equal(
    val_a[
        "mask"
    ],
    val_b[
        "mask"
    ]
)


# ============================================================
# BATCH SANITY
# ============================================================

batch = next(
    iter(
        dpssd_train_loader
    )
)

val_batch = next(
    iter(
        dpssd_val_loader
    )
)


print("\nDataset sizes:")
print(
    "Train:",
    len(
        dpssd_train_dataset
    )
)
print(
    "Val  :",
    len(
        dpssd_val_dataset
    )
)


print("\nTRAIN batch")
print(
    "Input :",
    tuple(
        batch[
            "input"
        ].shape
    )
)
print(
    "Target:",
    tuple(
        batch[
            "target"
        ].shape
    )
)
print(
    "Mask  :",
    tuple(
        batch[
            "mask"
        ].shape
    )
)
print(
    "Mask ratio:",
    f"{batch['mask'].float().mean().item()*100:.2f}%"
)


print("\nVAL batch")
print(
    "Input :",
    tuple(
        val_batch[
            "input"
        ].shape
    )
)
print(
    "Target:",
    tuple(
        val_batch[
            "target"
        ].shape
    )
)
print(
    "Mask  :",
    tuple(
        val_batch[
            "mask"
        ].shape
    )
)
print(
    "Mask ratio:",
    f"{val_batch['mask'].float().mean().item()*100:.2f}%"
)


# ------------------------------------------------------------
# Save frozen validation plan
# ------------------------------------------------------------

val_plan_path = (
    DPSSD_ROOT /
    "dpssd_fixed_internal_validation_plan_v1.csv"
)

dpssd_val_plan.to_csv(
    val_plan_path,
    index=False
)


print(
    "\nValidation deterministic: True"
)

print("\nSaved:")
print(val_plan_path)

print(
    "\nFinal DP-SSD data pipeline ready ✅"
)


## 4. DP-SSD model and sanity checks


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C23
# SECTION: P3-S8 — DP-SSD NETWORK
#
# PURPOSE:
#   Define the proposed lightweight dual-polarization
#   self-supervised despeckling network.
#
# DESIGN:
#   separate VV/VH stems
#       ↓
#   fused residual encoder
#       ↓
#   bottleneck
#       ↓
#   decoder with ordinary skip connections
#       ↓
#   2-channel clean log-SAR prediction
#
# NO attention / Transformer is used here.
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# BASIC RESIDUAL BLOCK
# ============================================================

class ResidualBlock(nn.Module):

    def __init__(
        self,
        channels
    ):

        super().__init__()

        self.conv1 = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            padding=1
        )

        self.conv2 = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            padding=1
        )

        self.act = nn.GELU()


    def forward(
        self,
        x
    ):

        residual = x

        x = self.conv1(x)
        x = self.act(x)

        x = self.conv2(x)

        x = x + residual

        return self.act(x)


# ============================================================
# ENCODER STAGE
# ============================================================

class EncoderStage(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.down = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=2,
            padding=1
        )

        self.block = ResidualBlock(
            out_channels
        )


    def forward(
        self,
        x
    ):

        x = self.down(x)
        x = self.block(x)

        return x


# ============================================================
# DECODER STAGE
# ============================================================

class DecoderStage(nn.Module):

    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels
    ):

        super().__init__()

        self.project = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            padding=1
        )

        self.fuse = nn.Conv2d(
            out_channels
            +
            skip_channels,
            out_channels,
            kernel_size=3,
            padding=1
        )

        self.block = ResidualBlock(
            out_channels
        )


    def forward(
        self,
        x,
        skip
    ):

        x = F.interpolate(
            x,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        x = self.project(x)

        x = torch.cat(
            [x, skip],
            dim=1
        )

        x = self.fuse(x)

        x = self.block(x)

        return x


# ============================================================
# DP-SSD
# ============================================================

class DPSSDNet(nn.Module):

    def __init__(
        self,
        stem_channels=16,
        base_channels=32
    ):

        super().__init__()


        # ----------------------------------------------------
        # Polarization-specific stems
        # ----------------------------------------------------

        self.vv_stem = nn.Sequential(

            nn.Conv2d(
                1,
                stem_channels,
                kernel_size=3,
                padding=1
            ),

            nn.GELU(),

            ResidualBlock(
                stem_channels
            )
        )


        self.vh_stem = nn.Sequential(

            nn.Conv2d(
                1,
                stem_channels,
                kernel_size=3,
                padding=1
            ),

            nn.GELU(),

            ResidualBlock(
                stem_channels
            )
        )


        # ----------------------------------------------------
        # Polarization fusion
        # ----------------------------------------------------

        self.fusion = nn.Sequential(

            nn.Conv2d(
                stem_channels * 2,
                base_channels,
                kernel_size=3,
                padding=1
            ),

            nn.GELU(),

            ResidualBlock(
                base_channels
            )
        )


        # ----------------------------------------------------
        # Encoder
        # ----------------------------------------------------

        self.enc1 = EncoderStage(
            base_channels,
            base_channels * 2
        )

        self.enc2 = EncoderStage(
            base_channels * 2,
            base_channels * 4
        )


        # ----------------------------------------------------
        # Bottleneck
        # ----------------------------------------------------

        self.bottleneck = nn.Sequential(

            ResidualBlock(
                base_channels * 4
            ),

            ResidualBlock(
                base_channels * 4
            )
        )


        # ----------------------------------------------------
        # Decoder
        # ----------------------------------------------------

        self.dec1 = DecoderStage(
            in_channels=base_channels * 4,
            skip_channels=base_channels * 2,
            out_channels=base_channels * 2
        )


        self.dec2 = DecoderStage(
            in_channels=base_channels * 2,
            skip_channels=base_channels,
            out_channels=base_channels
        )


        # ----------------------------------------------------
        # Output
        #
        # Direct clean normalized log-SAR prediction.
        # No global input→output residual shortcut.
        # ----------------------------------------------------

        self.output_head = nn.Conv2d(
            base_channels,
            2,
            kernel_size=1
        )


    def forward(
        self,
        x
    ):

        assert x.shape[1] == 2


        # Separate polarizations
        vv = x[:, 0:1]
        vh = x[:, 1:2]


        # Polarization-specific features
        vv = self.vv_stem(
            vv
        )

        vh = self.vh_stem(
            vh
        )


        # Fusion
        x0 = torch.cat(
            [vv, vh],
            dim=1
        )

        x0 = self.fusion(
            x0
        )


        # Encoder
        x1 = self.enc1(
            x0
        )

        x2 = self.enc2(
            x1
        )


        # Bottleneck
        x2 = self.bottleneck(
            x2
        )


        # Decoder
        x = self.dec1(
            x2,
            x1
        )

        x = self.dec2(
            x,
            x0
        )


        # Prediction
        out = self.output_head(
            x
        )

        return out


# ============================================================
# SANITY TEST
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = DPSSDNet().to(
    device
)


test_input = batch[
    "input"
].to(
    device
)


with torch.no_grad():

    test_output = model(
        test_input
    )


print("=" * 78)
print("P3-C23 — DP-SSD NETWORK SANITY TEST")
print("=" * 78)

print("Device      :", device)

print(
    "Input shape :",
    tuple(
        test_input.shape
    )
)

print(
    "Output shape:",
    tuple(
        test_output.shape
    )
)

print(
    "Output finite:",
    bool(
        torch.isfinite(
            test_output
        ).all()
    )
)


parameter_count = sum(
    p.numel()
    for p in model.parameters()
)

trainable_count = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print(
    f"\nParameters          : "
    f"{parameter_count:,}"
)

print(
    f"Trainable parameters: "
    f"{trainable_count:,}"
)


assert (
    test_output.shape
    ==
    test_input.shape
)

assert torch.isfinite(
    test_output
).all()


print(
    "\nDP-SSD architecture verified ✅"
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C24
# SECTION: P3-S8 — MASKED SELF-SUPERVISED LOSS
#
# PURPOSE:
#   Implement and verify the DP-SSD masked reconstruction loss.
#
# LOSS:
#   SmoothL1(prediction, target)
#   ONLY at intentionally hidden spatial positions.
#
# IMPORTANT:
#   The same spatial mask applies to VV and VH.
# ============================================================

import torch
import torch.nn.functional as F


# ============================================================
# MASKED SMOOTHL1 LOSS
# ============================================================

def masked_smooth_l1_loss(
    prediction,
    target,
    mask,
    beta=1.0
):
    """
    prediction : [B, 2, H, W]
    target     : [B, 2, H, W]
    mask       : [B, 1, H, W] boolean

    Loss is evaluated ONLY at masked pixels.
    """

    if prediction.shape != target.shape:
        raise ValueError(
            f"Prediction/target mismatch: "
            f"{prediction.shape} vs {target.shape}"
        )

    if mask.ndim != 4 or mask.shape[1] != 1:
        raise ValueError(
            f"Expected mask [B,1,H,W], got {mask.shape}"
        )

    # --------------------------------------------------------
    # Expand same spatial mask over VV and VH
    # --------------------------------------------------------

    mask_2ch = mask.expand(
        -1,
        prediction.shape[1],
        -1,
        -1
    )

    # --------------------------------------------------------
    # Pixel-wise SmoothL1
    # --------------------------------------------------------

    loss_map = F.smooth_l1_loss(
        prediction,
        target,
        reduction="none",
        beta=beta
    )

    masked_values = loss_map[
        mask_2ch
    ]

    if masked_values.numel() == 0:
        raise RuntimeError(
            "No masked pixels found in batch."
        )

    return masked_values.mean()


# ============================================================
# MOVE TEST BATCH + MODEL TO CURRENT DEVICE
# ============================================================

model = model.to(device)

input_batch = batch[
    "input"
].to(device)

target_batch = batch[
    "target"
].to(device)

mask_batch = batch[
    "mask"
].to(device)


# ============================================================
# FORWARD PASS
# ============================================================

model.train()

prediction = model(
    input_batch
)


# ============================================================
# MASKED LOSS
# ============================================================

loss = masked_smooth_l1_loss(
    prediction,
    target_batch,
    mask_batch
)


print("=" * 78)
print("P3-C24 — MASKED SELF-SUPERVISED LOSS")
print("=" * 78)

print(
    "Prediction shape:",
    tuple(prediction.shape)
)

print(
    "Target shape    :",
    tuple(target_batch.shape)
)

print(
    "Mask shape      :",
    tuple(mask_batch.shape)
)

print(
    "Mask ratio      :",
    f"{100 * mask_batch.float().mean().item():.2f}%"
)

print(
    "Masked loss     :",
    float(loss.item())
)


# ============================================================
# VERIFY THAT UNMASKED PIXELS DO NOT AFFECT LOSS
#
# We deliberately corrupt target values ONLY outside the mask.
# Correct implementation should produce identical loss.
# ============================================================

target_corrupted_unmasked = (
    target_batch.clone()
)

mask_2ch = mask_batch.expand(
    -1,
    2,
    -1,
    -1
)

target_corrupted_unmasked[
    ~mask_2ch
] += 1000.0


loss_after_unmasked_corruption = (
    masked_smooth_l1_loss(
        prediction,
        target_corrupted_unmasked,
        mask_batch
    )
)


difference = abs(
    loss.item()
    -
    loss_after_unmasked_corruption.item()
)


print(
    "\nLoss after corrupting UNMASKED targets:",
    float(
        loss_after_unmasked_corruption.item()
    )
)

print(
    "Difference:",
    difference
)


assert difference < 1e-7, (
    "Loss is incorrectly depending on "
    "unmasked pixels."
)


# ============================================================
# SECOND CHECK:
# Corrupt MASKED targets.
# Loss MUST change.
# ============================================================

target_corrupted_masked = (
    target_batch.clone()
)

target_corrupted_masked[
    mask_2ch
] += 5.0


loss_after_masked_corruption = (
    masked_smooth_l1_loss(
        prediction,
        target_corrupted_masked,
        mask_batch
    )
)


print(
    "\nLoss after corrupting MASKED targets:",
    float(
        loss_after_masked_corruption.item()
    )
)


assert not torch.isclose(
    loss,
    loss_after_masked_corruption
), (
    "Loss should change when masked "
    "targets are modified."
)


# ============================================================
# BACKPROP TEST
# ============================================================

model.zero_grad(
    set_to_none=True
)

loss.backward()


gradient_parameters = 0
finite_gradients = True

for p in model.parameters():

    if p.grad is not None:

        gradient_parameters += 1

        if not torch.isfinite(
            p.grad
        ).all():

            finite_gradients = False


print(
    "\nParameters receiving gradients:",
    gradient_parameters
)

print(
    "All gradients finite:",
    finite_gradients
)


assert gradient_parameters > 0
assert finite_gradients


print(
    "\nMasked self-supervised loss verified ✅"
)


In [ ]:
# ============================================================
# NOTEBOOK: 03_FloodShift_Preprocessing
# CELL: P3-C25
# SECTION: P3-S7 — DP-SSD TINY OVERFIT TEST
#
# PURPOSE:
#   Verify that DP-SSD can learn the masked reconstruction task
#   before launching a real training run.
#
# STRATEGY:
#   Train a fresh model repeatedly on ONE FIXED 4-sample batch.
#
# IMPORTANT:
#   This is a debugging sanity test, not a scientific result.
# ============================================================

import torch
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# Fresh model + fixed batch
# ------------------------------------------------------------

overfit_model = DPSSDNet().to(
    device
)

fixed_batch = next(
    iter(
        dpssd_train_loader
    )
)

fixed_input = fixed_batch[
    "input"
][:4].to(
    device
)

fixed_target = fixed_batch[
    "target"
][:4].to(
    device
)

fixed_mask = fixed_batch[
    "mask"
][:4].to(
    device
)


optimizer = torch.optim.AdamW(
    overfit_model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

N_STEPS = 200
loss_history = []


print("=" * 78)
print("P3-C25 — DP-SSD TINY OVERFIT TEST")
print("=" * 78)

print("Device :", device)
print("Steps  :", N_STEPS)
print(
    "Batch  :",
    tuple(
        fixed_input.shape
    )
)


overfit_model.train()

for step in range(
    1,
    N_STEPS + 1
):

    optimizer.zero_grad(
        set_to_none=True
    )

    prediction = overfit_model(
        fixed_input
    )

    loss = masked_smooth_l1_loss(
        prediction,
        fixed_target,
        fixed_mask
    )

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        overfit_model.parameters(),
        max_norm=5.0
    )

    optimizer.step()

    loss_history.append(
        float(
            loss.item()
        )
    )

    if (
        step == 1
        or
        step % 20 == 0
    ):

        print(
            f"Step {step:>3}/{N_STEPS}  "
            f"Loss = {loss.item():.6f}"
        )


initial_loss = loss_history[0]
final_loss = loss_history[-1]

reduction_pct = (
    100.0
    *
    (
        initial_loss
        -
        final_loss
    )
    /
    initial_loss
)


print("\n" + "-" * 78)

print(
    f"Initial loss : "
    f"{initial_loss:.6f}"
)

print(
    f"Final loss   : "
    f"{final_loss:.6f}"
)

print(
    f"Reduction    : "
    f"{reduction_pct:.2f}%"
)


plt.figure(
    figsize=(8, 4)
)

plt.plot(
    loss_history
)

plt.xlabel(
    "Optimization step"
)

plt.ylabel(
    "Masked SmoothL1 loss"
)

plt.title(
    "DP-SSD Tiny Overfit Test"
)

plt.grid(
    alpha=0.2
)

plt.show()


assert final_loss < initial_loss

assert torch.isfinite(
    torch.tensor(
        loss_history
    )
).all()


print(
    "\nDP-SSD can learn the masked reconstruction task ✅"
)
